In [1]:
import pandas as pd

df = pd.read_csv('bootstrap_samples_5000x30_1.csv')
percentile_df = pd.read_csv('percentile_1.csv')
df_params = pd.read_csv('data_fullX30_1.csv')

In [2]:
import re
import numpy as np
import pandas as pd

# percentile.csv 전처리 (한글 깨짐/표기 차이 보정)
score_table = percentile_df.copy()

def normalize_age_group(raw):
    if pd.isna(raw):
        return np.nan
    text = str(raw).strip().replace(' ', '')

    if text == '0-9':
        return '0-9'

    digits = re.findall(r'\d+', text)
    if not digits:
        return text

    decade = int(digits[0])
    if '+' in text or decade >= 80:
        return '80대+'
    return f'{decade}대'

def age_to_group(age):
    if pd.isna(age):
        return np.nan
    age = float(age)
    if age < 10:
        return '0-9'
    if age >= 80:
        return '80대+'
    return f'{int(age // 10) * 10}대'

score_table['age_group_norm'] = score_table['age_group'].apply(normalize_age_group)

# df 컬럼 기준 매핑
required_cols = ['RIDAGEYR', 'DAILY_SLEEP', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'avg_daily_steps']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f'df에 필요한 컬럼이 없습니다: {missing_cols}')

def sleep_deviation_from_daily_sleep(sleep_hours, age_group):
    age_sleep_ranges = {
        '0-9': (9.0, 11.0),
        '10대': (8.0, 10.0),
        '20대': (7.0, 9.0),
        '30대': (7.0, 9.0),
        '40대': (7.0, 9.0),
        '50대': (7.0, 9.0),
        '60대': (7.0, 9.0),
        '70대': (7.0, 8.0),
        '80대+': (7.0, 8.0),
    }
    low, high = age_sleep_ranges.get(age_group, (7.0, 9.0))
    if sleep_hours < low:
        return low - sleep_hours
    if sleep_hours > high:
        return sleep_hours - high
    return 0.0

def nearest_percentile(domain, age_group, value):
    sub = score_table[
        (score_table['domain'] == domain)
        & (score_table['age_group_norm'] == age_group)
        & (score_table['percentile_value'].notna())
    ][['percentile', 'percentile_value']].copy()

    if sub.empty:
        return np.nan, np.nan

    sub['diff'] = (sub['percentile_value'] - value).abs()
    row = sub.loc[sub['diff'].idxmin()]
    return int(row['percentile']), float(row['percentile_value'])

scored_df = df.copy()
scored_df['age_group_for_score'] = scored_df['RIDAGEYR'].apply(age_to_group)
scored_df['sleep_deviation_value'] = scored_df.apply(
    lambda row: sleep_deviation_from_daily_sleep(row['DAILY_SLEEP'], row['age_group_for_score']),
    axis=1,
)
scored_df['sleep_pattern_value'] = (
    scored_df['SLQ300']
    + scored_df['SLQ310'].abs()
    + scored_df['SLQ320']
    + scored_df['SLQ330']
)
scored_df['step_value'] = scored_df['avg_daily_steps']

# --- 기존 nearest_percentile 로직을 배치 계산으로 가속 (동률 시 첫 번째 선택 유지) ---
score_table_valid = score_table[score_table['percentile_value'].notna()].copy()

percentile_lookup = {}
for (domain, age_group), sub in score_table_valid.groupby(['domain', 'age_group_norm'], sort=False):
    percentile_lookup[(domain, age_group)] = (
        sub['percentile_value'].to_numpy(dtype=float),  # 원본 행 순서 유지
        sub['percentile'].to_numpy(dtype=float),
    )

def nearest_percentile_batch(domain, age_group_series, value_series):
    n = len(value_series)
    out_percentile = np.full(n, np.nan, dtype=float)
    out_ref_value = np.full(n, np.nan, dtype=float)

    age_arr = age_group_series.to_numpy()
    val_arr = value_series.to_numpy(dtype=float)

    for ag in pd.unique(age_group_series.dropna()):
        key = (domain, ag)
        if key not in percentile_lookup:
            continue

        ref_vals, ref_pcts = percentile_lookup[key]
        if ref_vals.size == 0:
            continue

        mask = (age_arr == ag) & np.isfinite(val_arr)
        if not np.any(mask):
            continue

        targets = val_arr[mask]
        diff = np.abs(targets[:, None] - ref_vals[None, :])
        idx = np.argmin(diff, axis=1)  # 최소값 동률 시 첫 번째 인덱스 선택

        out_percentile[mask] = ref_pcts[idx]
        out_ref_value[mask] = ref_vals[idx]

    return (
        pd.Series(out_percentile, index=value_series.index),
        pd.Series(out_ref_value, index=value_series.index),
    )

sleep_p, sleep_ref = nearest_percentile_batch(
    'sleep_time',
    scored_df['age_group_for_score'],
    scored_df['sleep_deviation_value'],
)
pattern_p, pattern_ref = nearest_percentile_batch(
    'sleep_pattern',
    scored_df['age_group_for_score'],
    scored_df['sleep_pattern_value'],
)
step_p, step_ref = nearest_percentile_batch(
    'step',
    scored_df['age_group_for_score'],
    scored_df['step_value'],
)

scored_df['sleep_percentile'] = sleep_p
scored_df['sleep_ref_value'] = sleep_ref
scored_df['sleep_pattern_percentile'] = pattern_p
scored_df['sleep_pattern_ref_value'] = pattern_ref
scored_df['step_percentile'] = step_p
scored_df['step_ref_value'] = step_ref

for percentile_col, score_col in [
    ('sleep_percentile', 'sleep_score'),
    ('sleep_pattern_percentile', 'sleep_pattern_score'),
    ('step_percentile', 'step_score'),
]:
    scored_df[score_col] = scored_df[percentile_col].apply(
        lambda p: np.nan if pd.isna(p) else max(0, 100 - int(p))
    )

scored_df['total_score'] = (
    scored_df['sleep_score']
    + scored_df['sleep_pattern_score']
    + scored_df['step_score']
)

score_cols = [
    'bootstrap_id', 'bootstrap_row_id', 'SEQN', 'RIDAGEYR', 'age_group_for_score',
    'DAILY_SLEEP', 'sleep_deviation_value', 'sleep_percentile', 'sleep_score',
    'sleep_pattern_value', 'sleep_pattern_percentile', 'sleep_pattern_score',
    'avg_daily_steps', 'step_percentile', 'step_score', 'total_score',
]

available_score_cols = [col for col in score_cols if col in scored_df.columns]
scored_df[available_score_cols].head(10)

,bootstrap_id,bootstrap_row_id,SEQN,RIDAGEYR,age_group_for_score,DAILY_SLEEP,sleep_deviation_value,sleep_percentile,sleep_score,sleep_pattern_value,sleep_pattern_percentile,sleep_pattern_score,avg_daily_steps,step_percentile,step_score,total_score
0,1,1,79005.0,44.0,40대,8.776401,0.000000,0.0,100,36.930408,90.0,10,25915.154170,0.0,100,210
1,1,2,79005.0,44.0,40대,7.274750,0.000000,0.0,100,22.873324,90.0,10,12082.359375,0.0,100,210
2,1,3,79005.0,44.0,40대,6.402836,0.597164,50.0,50,16.647151,90.0,10,26534.040105,0.0,100,160
3,1,4,79005.0,44.0,40대,6.906218,0.093782,10.0,90,22.835392,90.0,10,3321.503304,80.0,20,120
4,1,5,79005.0,44.0,40대,5.697430,1.302570,80.0,20,34.066591,90.0,10,1503.305247,90.0,10,40
5,1,6,79005.0,44.0,40대,10.535909,1.535909,80.0,20,44.105570,90.0,10,5250.221201,40.0,60,90
6,1,7,79005.0,44.0,40대,7.368770,0.000000,0.0,100,31.769998,90.0,10,4878.290505,50.0,50,160
7,1,8,79005.0,44.0,40대,8.493799,0.000000,0.0,100,36.902161,90.0,10,20820.899310,0.0,100,210
8,1,9,79005.0,44.0,40대,6.699903,0.300097,30.0,70,66.602476,90.0,10,2238.637330,90.0,10,90
9,1,10,79005.0,44.0,40대,5.917801,1.082199,70.0,30,34.473322,90.0,10,5535.326805,40.0,60,100


### 파라미터

In [3]:
# =========================
# 1) 입력 파라미터 (질환 4개 + 로지스틱 할인함수)
# =========================
P1 = 39380                    # 클러스터 1 기본보험료
p2 = 97580                    # 클러스터 2 기본보험료
p3 = 124990                   # 클러스터 3 기본보험료

# 로지스틱 할인함수 d(E) = L / (1 + exp(-k(E - E0)))
discount_L = 0.2              # 할인 상한 (비율)
discount_k = 0.03             # 할인 기울기
discount_E0 = 150.0           # 할인 중간점(노력 기준점)
max_discount_amount = 0.0     # 할인액 상한 (0 이하면 미적용)

# # 질환 4개(클러스터별 유병률)
# lambda1_1 = 0.62              # 심부전
# lambda1_2 = 0.84              # 관상동맥질환
# lambda1_3 = 0.59              # 협심증
# lambda1_4 = 1.08              # 심장마비
# lambda2_1 = 2.90              # 심부전
# lambda2_2 = 3.25              # 관상동맥질환
# lambda2_3 = 2.47              # 협심증
# lambda2_4 = 3.67              # 심장마비
# lambda3_1 = 7.39              # 심부전
# lambda3_2 = 8.85              # 관상동맥질환
# lambda3_3 = 5.60              # 협심증
# lambda3_4 = 8.24              # 심장마비

# alpha_1 = 2.01                # 전체 클러스터 평균 심부전
# alpha_2 = 2.3                 # 전체 클러스터 평균 관생동맥질환 
# alpha_3 = 1.71                # 전체 클러스터 평균 협심증
# alpha_4 = 2.63                # 전체 클러스터 평균 심장마비

# 질환 4개(클러스터별 유병률)
lambda1_1 = 0.62              # 심부전
lambda1_2 = 0.84              # 관상동맥질환
lambda1_3 = 0.59              # 협심증
lambda1_4 = 1.08              # 심장마비
lambda2_1 = 2.90              # 심부전
lambda2_2 = 3.25              # 관상동맥질환
lambda2_3 = 2.47              # 협심증
lambda2_4 = 3.67              # 심장마비
lambda3_1 = 7.39              # 심부전
lambda3_2 = 8.85              # 관상동맥질환
lambda3_3 = 5.60              # 협심증
lambda3_4 = 8.24              # 심장마비

alpha_1 = 2.01                # 전체 클러스터 평균 심부전
alpha_2 = 2.3                 # 전체 클러스터 평균 관생동맥질환 
alpha_3 = 1.71                # 전체 클러스터 평균 협심증
alpha_4 = 2.63                # 전체 클러스터 평균 심장마비

beta_1 = 763 / 100000 / 12    # 심부전의 월간 발생률
beta_2 = 103.5 / 100000 / 12  # 관상동맥질환의 월간 발생률
beta_3 = 107.6 / 100000 / 12  # 협심증의 월간 발생률
beta_4 = 68.2 / 100000 / 12   # 심장마비의 월간 발생률

# 노력 효과 계수
theta_step = 3
theta_sleep = 3
theta_sleep_pattern = 3

severity = 10000000.0         # S: 사고 1건당 평균 지급보험금 (1000만원)
operating_cost = 0.0          # C: 운영비


In [4]:
# =========================
# 2) 공통 함수/데이터 준비 (Cluster 기반)
# =========================
import numpy as np
import pandas as pd

def d_logistic(E, L, k, E0):
    E_arr = np.asarray(E, dtype=float)
    return np.clip(L / (1.0 + np.exp(-k * (E_arr - E0))), 0.0, 1.0)

def delta_effort(E, E0):
    E_arr = np.asarray(E, dtype=float)
    return np.maximum(0.0, E_arr - E0) / 300.0

# Cluster 컬럼 표준화 (cluster_id 사용 데이터도 허용)
if 'Cluster' not in scored_df.columns:
    if 'cluster_id' in scored_df.columns:
        scored_df['Cluster'] = scored_df['cluster_id']
    else:
        raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

# =========================
# 3) seqn_monthly_score 생성 및 보험료 계산
# =========================
required_scored_cols = ['SEQN', 'Cluster', 'total_score', 'sample_idx']
missing_scored_cols = [c for c in required_scored_cols if c not in scored_df.columns]
if missing_scored_cols:
    raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_scored_cols}")

# 1~30일 완전관측 기준 월평균 점수 생성
if 'bootstrap_id' in scored_df.columns:
    monthly_by_bootstrap = (
        scored_df[scored_df['sample_idx'].between(1, 30)]
        .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
        .agg(
            monthly_avg_total_score=('total_score', 'mean'),
            observed_days=('sample_idx', 'nunique')
        )
    )
    monthly_by_bootstrap = monthly_by_bootstrap[monthly_by_bootstrap['observed_days'] == 30].copy()
    seqn_monthly_score = (
        monthly_by_bootstrap
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(monthly_avg_total_score=('monthly_avg_total_score', 'mean'))
    )
else:
    seqn_monthly_score = (
        scored_df[scored_df['sample_idx'].between(1, 30)]
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(
            monthly_avg_total_score=('total_score', 'mean'),
            observed_days=('sample_idx', 'nunique')
        )
    )
    seqn_monthly_score = seqn_monthly_score[seqn_monthly_score['observed_days'] == 30].copy()

if seqn_monthly_score.empty:
    raise ValueError("완전관측(30일) 데이터가 없습니다.")

# 클러스터별 기본보험료
base_premium_by_cluster = {1: P1, 2: p2, 3: p3}

# 질병별 발생률: (클러스터 유병률 / 전체 클러스터 평균 유병률) * 월간 발생률
alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
prevalence_by_cluster = {
    1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
    2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
    3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
}

def cluster_occurrence_rate(prevalence_dict):
    disease_rates = [
        (prevalence_dict[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
        for idx in alpha_by_disease
    ]
    return float(np.sum(disease_rates))

lambda0_by_cluster = {
    cluster: cluster_occurrence_rate(prevalence_dict)
    for cluster, prevalence_dict in prevalence_by_cluster.items()
}

# 노력효과 계수 통합
theta = float(np.mean([theta_step, theta_sleep, theta_sleep_pattern]))

# 최적화 결과가 있으면 우선 적용, 없으면 기본 파라미터 사용
L_used = float(globals().get('L_opt', discount_L))
k_used = float(globals().get('k_opt', discount_k))
E0_used = float(globals().get('E0_opt', discount_E0))

pricing_df = seqn_monthly_score.copy()
pricing_df['base_premium_P0'] = pricing_df['Cluster'].map(base_premium_by_cluster)
pricing_df['lambda0'] = pricing_df['Cluster'].map(lambda0_by_cluster)

if pricing_df[['base_premium_P0', 'lambda0']].isna().any().any():
    unknown_clusters = sorted(pricing_df.loc[pricing_df['base_premium_P0'].isna(), 'Cluster'].dropna().unique().tolist())
    raise ValueError(f"파라미터에 없는 Cluster 값이 있습니다: {unknown_clusters}")

pricing_df['effort_score_E'] = pricing_df['monthly_avg_total_score'].clip(lower=0, upper=300)
pricing_df['discount_rate'] = d_logistic(pricing_df['effort_score_E'], L_used, k_used, E0_used)
pricing_df['raw_discount_amount'] = pricing_df['base_premium_P0'] * pricing_df['discount_rate']

if max_discount_amount > 0:
    pricing_df['applied_discount_amount'] = pricing_df['raw_discount_amount'].clip(upper=max_discount_amount)
else:
    pricing_df['applied_discount_amount'] = pricing_df['raw_discount_amount']

pricing_df['premium_P'] = pricing_df['base_premium_P0'] - pricing_df['applied_discount_amount']
pricing_df['delta_E'] = delta_effort(pricing_df['effort_score_E'], E0_used)
pricing_df['lambda_effective'] = pricing_df['lambda0'] * np.exp(-theta * pricing_df['delta_E'])
pricing_df['expected_loss_E_L'] = pricing_df['lambda_effective'] * severity
pricing_df['profit_Pi'] = pricing_df['premium_P'] - pricing_df['expected_loss_E_L'] - operating_cost

seqn_premium = (
    pricing_df[
        [
            'SEQN', 'Cluster', 'monthly_avg_total_score',
            'effort_score_E', 'base_premium_P0', 'discount_rate',
            'premium_P', 'lambda0', 'lambda_effective',
            'expected_loss_E_L', 'profit_Pi'
        ]
    ]
    .sort_values(['Cluster', 'SEQN'])
    .reset_index(drop=True)
)

print(f"사용 할인 파라미터: L={L_used:.4f}, k={k_used:.6f}, E0={E0_used:.2f}")
seqn_premium.head(10)

사용 할인 파라미터: L=0.2000, k=0.030000, E0=150.00


,SEQN,Cluster,monthly_avg_total_score,effort_score_E,base_premium_P0,discount_rate,premium_P,lambda0,lambda_effective,expected_loss_E_L,profit_Pi
0,62301.0,1,150.369267,150.369267,39380,0.100554,35420.187641,0.000282,0.000281,2808.646629,32611.541012
1,62371.0,1,149.447867,149.447867,39380,0.099172,35474.613770,0.000282,0.000282,2819.037197,32655.576573
2,62479.0,1,149.363800,149.363800,39380,0.099046,35479.579193,0.000282,0.000282,2819.037197,32660.541996
3,62771.0,1,149.232267,149.232267,39380,0.098848,35487.348003,0.000282,0.000282,2819.037197,32668.310806
4,62884.0,1,149.480800,149.480800,39380,0.099221,35472.668524,0.000282,0.000282,2819.037197,32653.631327
5,63261.0,1,145.527467,145.527467,39380,0.093301,35705.796897,0.000282,0.000282,2819.037197,32886.759700
6,63793.0,1,150.249400,150.249400,39380,0.100374,35427.268011,0.000282,0.000281,2812.015279,32615.252732
7,63882.0,1,145.731400,145.731400,39380,0.093606,35693.802190,0.000282,0.000282,2819.037197,32874.764993
8,64018.0,1,149.355333,149.355333,39380,0.099033,35480.079273,0.000282,0.000282,2819.037197,32661.042076
9,64133.0,1,149.541200,149.541200,39380,0.099312,35469.100888,0.000282,0.000282,2819.037197,32650.063691


### MLE 최적화

In [5]:
# 노력점수 분포 적합 MLE (절단 로지스틱: 0~300 구간)
import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution
from scipy.special import expit

# -----------------------------
# 0) df_params를 df와 동일 로직으로 스코어링
# -----------------------------
required_globals = ['score_table', 'age_to_group', 'sleep_deviation_from_daily_sleep', 'nearest_percentile_batch']
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise ValueError(
        f"전처리 함수/테이블이 없습니다: {missing_globals}. 먼저 df 전처리 셀을 실행하세요."
    )

required_df_params_cols = [
    'RIDAGEYR', 'DAILY_SLEEP', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'avg_daily_steps',
    'SEQN'
 ]
missing_df_params_cols = [c for c in required_df_params_cols if c not in df_params.columns]
if missing_df_params_cols:
    raise ValueError(f"df_params에 필요한 컬럼이 없습니다: {missing_df_params_cols}")

scored_df_params = df_params.copy()
scored_df_params['age_group_for_score'] = scored_df_params['RIDAGEYR'].apply(age_to_group)
scored_df_params['sleep_deviation_value'] = scored_df_params.apply(
    lambda row: sleep_deviation_from_daily_sleep(row['DAILY_SLEEP'], row['age_group_for_score']),
    axis=1,
 )
scored_df_params['sleep_pattern_value'] = (
    scored_df_params['SLQ300']
    + scored_df_params['SLQ310'].abs()
    + scored_df_params['SLQ320']
    + scored_df_params['SLQ330']
)
scored_df_params['step_value'] = scored_df_params['avg_daily_steps']

sleep_p, sleep_ref = nearest_percentile_batch(
    'sleep_time',
    scored_df_params['age_group_for_score'],
    scored_df_params['sleep_deviation_value'],
)
pattern_p, pattern_ref = nearest_percentile_batch(
    'sleep_pattern',
    scored_df_params['age_group_for_score'],
    scored_df_params['sleep_pattern_value'],
)
step_p, step_ref = nearest_percentile_batch(
    'step',
    scored_df_params['age_group_for_score'],
    scored_df_params['step_value'],
)

scored_df_params['sleep_percentile'] = sleep_p
scored_df_params['sleep_ref_value'] = sleep_ref
scored_df_params['sleep_pattern_percentile'] = pattern_p
scored_df_params['sleep_pattern_ref_value'] = pattern_ref
scored_df_params['step_percentile'] = step_p
scored_df_params['step_ref_value'] = step_ref

for percentile_col, score_col in [
    ('sleep_percentile', 'sleep_score'),
    ('sleep_pattern_percentile', 'sleep_pattern_score'),
    ('step_percentile', 'step_score'),
]:
    scored_df_params[score_col] = scored_df_params[percentile_col].apply(
        lambda p: np.nan if pd.isna(p) else max(0, 100 - int(p))
    )

scored_df_params['total_score'] = (
    scored_df_params['sleep_score']
    + scored_df_params['sleep_pattern_score']
    + scored_df_params['step_score']
)

# Cluster 컬럼 표준화
if 'Cluster' not in scored_df_params.columns:
    if 'cluster_id' in scored_df_params.columns:
        scored_df_params['Cluster'] = scored_df_params['cluster_id']
    else:
        raise ValueError("df_params(→scored_df_params)에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

# -----------------------------
# 1) 월간(1~30일) 완전관측 집계 (df_params 기반)
# -----------------------------
if 'sample_idx' in scored_df_params.columns:
    monthly = (
        scored_df_params[scored_df_params['sample_idx'].between(1, 30)]
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(
            monthly_avg_total_score=('total_score', 'mean'),
            observed_days=('sample_idx', 'nunique')
        )
    )
    monthly = monthly[monthly['observed_days'] == 30].copy()
else:
    monthly = (
        scored_df_params
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(monthly_avg_total_score=('total_score', 'mean'))
    )
    monthly['observed_days'] = 30

if monthly.empty:
    raise ValueError("df_params 기반 완전관측(30일) 데이터가 없습니다. df_params를 확인하세요.")

# 2) 노력점수 벡터 준비 (df_params 기반)
E_vals = monthly['monthly_avg_total_score'].clip(0.0, 300.0).to_numpy(dtype=float)
if E_vals.size < 10:
    raise ValueError("MLE 적합을 위한 표본 수가 너무 적습니다(최소 10개 권장).")

LOW, HIGH = 0.0, 300.0
FULL_SCORE = 300.0
FIXED_FULL_SCORE_DISCOUNT_RATE = discount_L # 300점 고객 할인율을 5%로 고정
EPS = 1e-12

# 3) 절단 로지스틱 분포 음의 로그우도
#    X ~ Logistic(E0, scale=1/k), X in [0, 300] 조건부 분포
#    pdf(x) = [k*s(x)*(1-s(x))] / [S(300)-S(0)], s(x)=sigmoid(k*(x-E0))
def neg_log_likelihood(theta):
    log_k, E0 = theta
    k = float(np.exp(log_k))

    s = expit(k * (E_vals - E0))
    base_pdf = k * s * (1.0 - s)

    cdf_low = expit(k * (LOW - E0))
    cdf_high = expit(k * (HIGH - E0))
    norm_const = cdf_high - cdf_low

    if (not np.isfinite(norm_const)) or (norm_const <= EPS):
        return np.inf

    ll = np.log(np.clip(base_pdf, EPS, None)) - np.log(np.clip(norm_const, EPS, None))
    return -float(np.sum(ll))

# 4) 전역 + 로컬 최적화 (k>0 제약은 log_k로 처리)
#    k in [1e-4, 1.0], E0 in [0, 300]
bounds = [
    (np.log(1e-4), np.log(1.0)),
    (LOW, HIGH),
]

de_res = differential_evolution(neg_log_likelihood, bounds=bounds, seed=42, maxiter=200)
res = minimize(neg_log_likelihood, de_res.x, bounds=bounds, method='L-BFGS-B')
best_x = res.x if res.success else de_res.x

log_k_opt, E0_opt = [float(v) for v in best_x]
k_opt = float(np.exp(log_k_opt))

# 300점 할인율을 정확히 5%로 맞추기 위해 L 역산
# d(E)=L/(1+exp(-k(E-E0))) 이므로 d(300)=0.05가 되게 L 계산
sigmoid_full = float(expit(k_opt * (FULL_SCORE - E0_opt)))
if sigmoid_full <= EPS:
    raise ValueError("추정된 k/E0로는 300점 기준 정규화가 불안정합니다.")

L_opt = float(FIXED_FULL_SCORE_DISCOUNT_RATE / sigmoid_full)

# downstream 일관 적용 (비교 시뮬레이션은 df 기반 scored_df를 그대로 사용)
discount_L = L_opt
discount_k = k_opt
discount_E0 = E0_opt

# 5) 적합도 요약 (KS distance)
def truncated_logistic_cdf(x, k, E0, low=0.0, high=300.0):
    f_low = expit(k * (low - E0))
    f_high = expit(k * (high - E0))
    denom = np.clip(f_high - f_low, EPS, None)
    fx = expit(k * (x - E0))
    out = (fx - f_low) / denom
    return np.clip(out, 0.0, 1.0)

E_sorted = np.sort(E_vals)
n = E_sorted.size
emp_cdf = np.arange(1, n + 1, dtype=float) / n
fit_cdf = truncated_logistic_cdf(E_sorted, k_opt, E0_opt, LOW, HIGH)
ks_distance = float(np.max(np.abs(emp_cdf - fit_cdf)))

nll_opt = float(neg_log_likelihood([log_k_opt, E0_opt]))
aic = 2 * 2 + 2 * nll_opt  # 파라미터 2개(k, E0)

d300_check = float(np.clip(L_opt / (1.0 + np.exp(-k_opt * (FULL_SCORE - E0_opt))), 0.0, 1.0))

print("=== 노력점수 분포 MLE (절단 로지스틱) 결과: df_params 기반 ===")
print(f"표본 수 n = {n}")
print(f"300점 고정 할인율 = {FIXED_FULL_SCORE_DISCOUNT_RATE:.2%}")
print(f"L(역산)  = {L_opt:.6f}")
print(f"k(MLE)   = {k_opt:.6f}")
print(f"E0(MLE)  = {E0_opt:.3f}")
print(f"d(300) 확인 = {d300_check:.4%}")
print(f"NLL      = {nll_opt:.2f}")
print(f"AIC      = {aic:.2f}")
print(f"KS dist  = {ks_distance:.4f}")

monthly_sample = monthly.copy()
monthly_sample['fitted_cdf'] = truncated_logistic_cdf(
    monthly_sample['monthly_avg_total_score'].clip(LOW, HIGH).to_numpy(dtype=float),
    k_opt,
    E0_opt,
    LOW,
    HIGH,
)

monthly_sample[
    [
        'SEQN', 'Cluster', 'monthly_avg_total_score',
        'fitted_cdf'
    ]
] .head(10)

=== 노력점수 분포 MLE (절단 로지스틱) 결과: df_params 기반 ===
표본 수 n = 7382
300점 고정 할인율 = 20.00%
L(역산)  = 0.200294
k(MLE)   = 0.048319
E0(MLE)  = 165.046
d(300) 확인 = 20.0000%
NLL      = 37436.85
AIC      = 74877.69
KS dist  = 0.2196


,SEQN,Cluster,monthly_avg_total_score,fitted_cdf
0,62161.0,1,160.333333,0.443780
1,62164.0,1,154.333333,0.373744
2,62169.0,1,152.666667,0.355068
3,62172.0,1,136.333333,0.199850
4,62177.0,2,159.333333,0.431869
5,62180.0,1,147.000000,0.295040
6,62184.0,1,149.000000,0.315559
7,62189.0,1,148.666667,0.312086
8,62199.0,2,151.666667,0.344067
9,62200.0,2,153.000000,0.358771


### 비교 시뮬레이션

In [6]:
# ==========================================
# 보험료 비교: (1) 클러스터만 vs (2) 클러스터+월간 total_score(로지스틱 할인)
# + bootstrap_id 단위 통계적 유의성 검정
# ==========================================
import numpy as np
import pandas as pd
from scipy import stats

# ---- 0) 입력 검증 ----
if 'Cluster' not in scored_df.columns:
    if 'cluster_id' in scored_df.columns:
        scored_df['Cluster'] = scored_df['cluster_id']
    else:
        raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

required_cols_compare = ['bootstrap_id', 'sample_idx', 'SEQN', 'Cluster', 'total_score']
missing_compare_cols = [c for c in required_cols_compare if c not in scored_df.columns]
if missing_compare_cols:
    raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_compare_cols}")

required_params = ['P1', 'p2', 'p3', 'max_discount_amount', 'discount_L', 'discount_k', 'discount_E0']
for pname in required_params:
    if pname not in globals():
        raise ValueError(f"파라미터 셀을 먼저 실행하세요. 누락: {pname}")

def d_logistic(E, L, k, E0):
    E_arr = np.asarray(E, dtype=float)
    return np.clip(L / (1.0 + np.exp(-k * (E_arr - E0))), 0.0, 1.0)

base_premium_by_cluster = {1: P1, 2: p2, 3: p3}

# ---- 1) bootstrap_id별 월간 점수(1~30일) 집계 ----
monthly_by_bootstrap = (
    scored_df[scored_df['sample_idx'].between(1, 30)]
    .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
    .agg(
        monthly_avg_total_score=('total_score', 'mean'),
        observed_days=('sample_idx', 'nunique')
    )
)
monthly_by_bootstrap = monthly_by_bootstrap[monthly_by_bootstrap['observed_days'] == 30].copy()

# ---- 2) 두 방식 보험료 계산 ----
compare_df = monthly_by_bootstrap.copy()
compare_df['base_premium_P0'] = compare_df['Cluster'].map(base_premium_by_cluster)

if compare_df['base_premium_P0'].isna().any():
    unknown_clusters = sorted(compare_df.loc[compare_df['base_premium_P0'].isna(), 'Cluster'].dropna().unique().tolist())
    raise ValueError(f"기본보험료 파라미터에 없는 Cluster 값이 있습니다: {unknown_clusters}")

# 방식 A: 클러스터만 사용
compare_df['premium_cluster_only'] = compare_df['base_premium_P0']

# 방식 B: 클러스터 + 월간 total_score + 로지스틱 할인
compare_df['effort_score_E'] = compare_df['monthly_avg_total_score'].clip(lower=0, upper=300)
compare_df['discount_rate'] = d_logistic(compare_df['effort_score_E'], discount_L, discount_k, discount_E0)
compare_df['raw_discount_amount'] = compare_df['base_premium_P0'] * compare_df['discount_rate']

if max_discount_amount > 0:
    compare_df['applied_discount_amount'] = compare_df['raw_discount_amount'].clip(upper=max_discount_amount)
else:
    compare_df['applied_discount_amount'] = compare_df['raw_discount_amount']

compare_df['premium_with_monthly_score'] = compare_df['base_premium_P0'] - compare_df['applied_discount_amount']

# ---- 3) bootstrap_id 단위 요약(평균 보험료) ----
bootstrap_summary = (
    compare_df
    .groupby('bootstrap_id', as_index=False)
    .agg(
        n_seqn=('SEQN', 'nunique'),
        mean_premium_cluster_only=('premium_cluster_only', 'mean'),
        mean_premium_with_score=('premium_with_monthly_score', 'mean')
    )
)
bootstrap_summary['mean_diff_score_minus_cluster'] = (
    bootstrap_summary['mean_premium_with_score'] - bootstrap_summary['mean_premium_cluster_only']
)

# ---- 4) 통계 검정 ----
diffs = bootstrap_summary['mean_diff_score_minus_cluster'].to_numpy(dtype=float)
if len(diffs) < 2:
    raise ValueError('통계 검정을 위해 bootstrap_id가 최소 2개 이상 필요합니다.')

ttest_res = stats.ttest_rel(
    bootstrap_summary['mean_premium_with_score'],
    bootstrap_summary['mean_premium_cluster_only']
)

try:
    wilcoxon_res = stats.wilcoxon(
        bootstrap_summary['mean_premium_with_score'],
        bootstrap_summary['mean_premium_cluster_only'],
        zero_method='wilcox',
        alternative='two-sided'
    )
    wilcoxon_stat = float(wilcoxon_res.statistic)
    wilcoxon_p = float(wilcoxon_res.pvalue)
except ValueError:
    wilcoxon_stat = np.nan
    wilcoxon_p = np.nan

rng = np.random.default_rng(42)
B = 5000
boot_means = np.empty(B)
for i in range(B):
    sample = rng.choice(diffs, size=len(diffs), replace=True)
    boot_means[i] = sample.mean()
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

stats_result = pd.DataFrame([
    {
        'n_bootstrap_id': int(len(diffs)),
        'mean_diff(score-cluster)': float(diffs.mean()),
        'bootstrap95ci_low': float(ci_low),
        'bootstrap95ci_high': float(ci_high),
        'paired_t_stat': float(ttest_res.statistic),
        'paired_t_pvalue': float(ttest_res.pvalue),
        'wilcoxon_stat': wilcoxon_stat,
        'wilcoxon_pvalue': wilcoxon_p,
    }
])

print('=== bootstrap_id별 평균 보험료 비교 ===')
display(bootstrap_summary)

print('\n=== 통계 검정 결과 ===')
display(stats_result)

seqn_premium_compare = compare_df[
    [
        'bootstrap_id', 'SEQN', 'Cluster', 'monthly_avg_total_score',
        'premium_cluster_only', 'premium_with_monthly_score'
    ]
] .sort_values(['bootstrap_id', 'Cluster', 'SEQN']).reset_index(drop=True)

print('\n=== SEQN별 보험료(일부) ===')
seqn_premium_compare.head(20)

=== bootstrap_id별 평균 보험료 비교 ===


,bootstrap_id,n_seqn,mean_premium_cluster_only,mean_premium_with_score,mean_diff_score_minus_cluster
0,1,100,62216.9,56325.922683,-5890.977317
1,2,100,62216.9,56669.931171,-5546.968829
2,3,100,62216.9,56536.793171,-5680.106829
3,4,100,62216.9,56651.938002,-5564.961998
4,5,100,62216.9,56596.867476,-5620.032524
...,...,...,...,...,...
4995,4996,100,62216.9,56453.638483,-5763.261517
4996,4997,100,62216.9,56646.795434,-5570.104566
4997,4998,100,62216.9,56472.022788,-5744.877212
4998,4999,100,62216.9,56363.091673,-5853.808327



=== 통계 검정 결과 ===


,n_bootstrap_id,mean_diff(score-cluster),bootstrap95ci_low,bootstrap95ci_high,paired_t_stat,paired_t_pvalue,wilcoxon_stat,wilcoxon_pvalue
0,5000,-5757.236448,-5760.310825,-5754.190086,-3692.388156,0.0,0.0,0.0



=== SEQN별 보험료(일부) ===


,bootstrap_id,SEQN,Cluster,monthly_avg_total_score,premium_cluster_only,premium_with_monthly_score
0,1,62301.0,1,145.000000,39380,37209.633758
1,1,62371.0,1,147.666667,39380,37001.181113
2,1,62479.0,1,143.000000,39380,37358.314087
3,1,62771.0,1,139.333333,39380,37613.024931
4,1,62884.0,1,148.000000,39380,36974.337149
5,1,63261.0,1,135.333333,39380,37863.870918
6,1,63793.0,1,129.333333,39380,38187.761299
7,1,63882.0,1,139.000000,39380,37635.010605
8,1,64018.0,1,146.666667,39380,37080.679780
9,1,64133.0,1,150.666667,39380,36753.666819


In [7]:
# ==========================================
# 보험사 지출 예상 비교: (1) 클러스터만 vs (2) 클러스터+월간 total_score
# ==========================================
import numpy as np
import pandas as pd
from scipy import stats

# ---- 0) 입력 검증 ----
if 'Cluster' not in scored_df.columns:
    if 'cluster_id' in scored_df.columns:
        scored_df['Cluster'] = scored_df['cluster_id']
    else:
        raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

required_cols_spend = ['bootstrap_id', 'sample_idx', 'SEQN', 'Cluster', 'total_score']
missing_spend_cols = [c for c in required_cols_spend if c not in scored_df.columns]
if missing_spend_cols:
    raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_spend_cols}")

required_spend_params = [
    'theta_step', 'theta_sleep', 'theta_sleep_pattern', 'severity', 'operating_cost',
    'discount_E0',
    'alpha_1', 'alpha_2', 'alpha_3', 'alpha_4',
    'beta_1', 'beta_2', 'beta_3', 'beta_4',
    'lambda1_1', 'lambda1_2', 'lambda1_3', 'lambda1_4',
    'lambda2_1', 'lambda2_2', 'lambda2_3', 'lambda2_4',
    'lambda3_1', 'lambda3_2', 'lambda3_3', 'lambda3_4',
]
for pname in required_spend_params:
    if pname not in globals():
        raise ValueError(f"파라미터 셀을 먼저 실행하세요. 누락: {pname}")

# 도메인별 노력효과 계수 통합(평균)
theta = float(np.mean([theta_step, theta_sleep, theta_sleep_pattern]))
effort_threshold = float(np.clip(discount_E0, 0.0, 299.999))

# ---- 1) 질병별 발생률 기반 클러스터 λ0 매핑 ----
alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
prevalence_by_cluster = {
    1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
    2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
    3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
}
lambda0_by_cluster_spend = {
    cluster: float(np.sum([
        (prev_by_disease[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
        for idx in alpha_by_disease
    ]))
    for cluster, prev_by_disease in prevalence_by_cluster.items()
}

# ---- 2) bootstrap_id별 월간 점수(1~30일) 집계 ----
monthly_by_bootstrap_spend = (
    scored_df[scored_df['sample_idx'].between(1, 30)]
    .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
    .agg(
        monthly_avg_total_score=('total_score', 'mean'),
        observed_days=('sample_idx', 'nunique')
    )
)
monthly_by_bootstrap_spend = monthly_by_bootstrap_spend[monthly_by_bootstrap_spend['observed_days'] == 30].copy()

# ---- 3) 두 방식 지출 계산 ----
spend_df = monthly_by_bootstrap_spend.copy()
spend_df['lambda0'] = spend_df['Cluster'].map(lambda0_by_cluster_spend)
if spend_df['lambda0'].isna().any():
    unknown_clusters = sorted(spend_df.loc[spend_df['lambda0'].isna(), 'Cluster'].dropna().unique().tolist())
    raise ValueError(f"유병률 파라미터에 없는 Cluster 값이 있습니다: {unknown_clusters}")

# 방식 A: 클러스터만 사용
spend_df['expected_expenditure_cluster_only'] = spend_df['lambda0'] * severity + operating_cost

# 방식 B: 클러스터 + 월간 total_score 사용
spend_df['effort_score_E'] = spend_df['monthly_avg_total_score'].clip(lower=0, upper=300)
spend_df['delta_E'] = np.maximum(0.0, spend_df['effort_score_E'] - effort_threshold) / 300.0
spend_df['lambda_effective'] = spend_df['lambda0'] * np.exp(-theta * spend_df['delta_E'])
spend_df['expected_expenditure_with_score'] = spend_df['lambda_effective'] * severity + operating_cost

# ---- 4) bootstrap_id 단위 요약(평균 지출)
spend_bootstrap_summary = (
    spend_df
    .groupby('bootstrap_id', as_index=False)
    .agg(
        n_seqn=('SEQN', 'nunique'),
        mean_exp_cluster_only=('expected_expenditure_cluster_only', 'mean'),
        mean_exp_with_score=('expected_expenditure_with_score', 'mean')
    )
)
spend_bootstrap_summary['mean_diff_with_score_minus_cluster'] = (
    spend_bootstrap_summary['mean_exp_with_score'] - spend_bootstrap_summary['mean_exp_cluster_only']
)

# ---- 5) 통계 검정 ----
spend_diffs = spend_bootstrap_summary['mean_diff_with_score_minus_cluster'].to_numpy(dtype=float)
if len(spend_diffs) < 2:
    raise ValueError('통계 검정을 위해 bootstrap_id가 최소 2개 이상 필요합니다.')

spend_ttest_res = stats.ttest_rel(
    spend_bootstrap_summary['mean_exp_with_score'],
    spend_bootstrap_summary['mean_exp_cluster_only']
)

try:
    spend_wilcoxon_res = stats.wilcoxon(
        spend_bootstrap_summary['mean_exp_with_score'],
        spend_bootstrap_summary['mean_exp_cluster_only'],
        zero_method='wilcox',
        alternative='two-sided'
    )
    spend_wilcoxon_stat = float(spend_wilcoxon_res.statistic)
    spend_wilcoxon_p = float(spend_wilcoxon_res.pvalue)
except ValueError:
    spend_wilcoxon_stat = np.nan
    spend_wilcoxon_p = np.nan

rng = np.random.default_rng(42)
B = 5000
spend_boot_means = np.empty(B)
for i in range(B):
    sample = rng.choice(spend_diffs, size=len(spend_diffs), replace=True)
    spend_boot_means[i] = sample.mean()
spend_ci_low, spend_ci_high = np.percentile(spend_boot_means, [2.5, 97.5])

spend_stats_result = pd.DataFrame([
    {
        'n_bootstrap_id': int(len(spend_diffs)),
        'mean_diff(with_score-cluster)': float(spend_diffs.mean()),
        'bootstrap95ci_low': float(spend_ci_low),
        'bootstrap95ci_high': float(spend_ci_high),
        'paired_t_stat': float(spend_ttest_res.statistic),
        'paired_t_pvalue': float(spend_ttest_res.pvalue),
        'wilcoxon_stat': spend_wilcoxon_stat,
        'wilcoxon_pvalue': spend_wilcoxon_p,
    }
])

print('=== bootstrap_id별 보험사 예상 지출 비교 ===')
display(spend_bootstrap_summary)

print('\n=== 통계 검정 결과 ===')
display(spend_stats_result)

spend_seqn_compare = spend_df[
    [
        'bootstrap_id', 'SEQN', 'Cluster', 'monthly_avg_total_score',
        'expected_expenditure_cluster_only', 'expected_expenditure_with_score'
    ]
] .sort_values(['bootstrap_id', 'Cluster', 'SEQN']).reset_index(drop=True)

print('\n=== SEQN별 예상 지출(일부) ===')
spend_seqn_compare.head(20)

=== bootstrap_id별 보험사 예상 지출 비교 ===


,bootstrap_id,n_seqn,mean_exp_cluster_only,mean_exp_with_score,mean_diff_with_score_minus_cluster
0,1,100,7904.531543,6656.321966,-1248.209576
1,2,100,7904.531543,6665.528349,-1239.003194
2,3,100,7904.531543,6681.662917,-1222.868625
3,4,100,7904.531543,6683.152128,-1221.379414
4,5,100,7904.531543,6657.117247,-1247.414296
...,...,...,...,...,...
4995,4996,100,7904.531543,6673.837832,-1230.693710
4996,4997,100,7904.531543,6666.439226,-1238.092316
4997,4998,100,7904.531543,6666.671836,-1237.859707
4998,4999,100,7904.531543,6659.454809,-1245.076734



=== 통계 검정 결과 ===


,n_bootstrap_id,mean_diff(with_score-cluster),bootstrap95ci_low,bootstrap95ci_high,paired_t_stat,paired_t_pvalue,wilcoxon_stat,wilcoxon_pvalue
0,5000,-1232.671158,-1233.39844,-1231.873836,-3075.005347,0.0,0.0,0.0



=== SEQN별 예상 지출(일부) ===


,bootstrap_id,SEQN,Cluster,monthly_avg_total_score,expected_expenditure_cluster_only,expected_expenditure_with_score
0,1,62301.0,1,145.000000,2819.037197,2819.037197
1,1,62371.0,1,147.666667,2819.037197,2819.037197
2,1,62479.0,1,143.000000,2819.037197,2819.037197
3,1,62771.0,1,139.333333,2819.037197,2819.037197
4,1,62884.0,1,148.000000,2819.037197,2819.037197
5,1,63261.0,1,135.333333,2819.037197,2819.037197
6,1,63793.0,1,129.333333,2819.037197,2819.037197
7,1,63882.0,1,139.000000,2819.037197,2819.037197
8,1,64018.0,1,146.666667,2819.037197,2819.037197
9,1,64133.0,1,150.666667,2819.037197,2819.037197


In [8]:
# === 클러스터별 경험적 평균 할인율 계산 (실제 데이터 기반) ===
import numpy as np
import pandas as pd

# 전제: 이전 셀에서 `compare_df`(월간 집계, discount_rate 포함)를 생성했음
if 'compare_df' not in globals():
    raise ValueError("`compare_df`가 없습니다. 먼저 '보험료 비교' 셀을 실행하세요.")

# discount 관련 컬럼이 없으면 재계산(안전장치)
if 'discount_rate' not in compare_df.columns or 'applied_discount_amount' not in compare_df.columns:
    compare_df['effort_score_E'] = compare_df['monthly_avg_total_score'].clip(lower=0, upper=300)
    compare_df['discount_rate'] = d_logistic(compare_df['effort_score_E'], discount_L, discount_k, discount_E0)
    compare_df['raw_discount_amount'] = compare_df['base_premium_P0'] * compare_df['discount_rate']
    compare_df['applied_discount_amount'] = (
        compare_df['raw_discount_amount'].clip(upper=max_discount_amount)
        if max_discount_amount > 0
        else compare_df['raw_discount_amount']
    )

# 적용 할인 비율(보험료 대비 실제 할인 비율)
compare_df['applied_discount_pct'] = compare_df['applied_discount_amount'] / compare_df['base_premium_P0']

# 클러스터별 관측치 기반 요약(경험적 평균 할인율)
cluster_emp = (
    compare_df
    .groupby('Cluster', as_index=False)
    .agg(
        n_seqn=('SEQN', 'nunique'),
        mean_discount_rate=('discount_rate', 'mean'),
        median_discount_rate=('discount_rate', 'median'),
        std_discount_rate=('discount_rate', 'std'),
        mean_applied_discount_pct=('applied_discount_pct', 'mean'),
    )
)

# 부트스트랩 단위(있으면)로 클러스터별 평균과 95% CI 계산
if 'bootstrap_id' in compare_df.columns:
    boot_cluster = (
        compare_df
        .groupby(['bootstrap_id', 'Cluster'], as_index=False)['discount_rate']
        .mean()
    )
    ci = (
        boot_cluster
        .groupby('Cluster')['discount_rate']
        .quantile([0.025, 0.975])
        .unstack(level=1)
        .rename(columns={0.025: 'ci_low', 0.975: 'ci_high'})
        .reset_index()
    )

    boot_mean = (
        boot_cluster.groupby('Cluster', as_index=False)['discount_rate'].mean()
        .rename(columns={'discount_rate': 'boot_mean_discount_rate'})
    )

    cluster_emp = (
        cluster_emp.merge(boot_mean, on='Cluster', how='left')
                   .merge(ci, on='Cluster', how='left')
    )

# 보기 좋게 퍼센트 컬럼 추가
cluster_emp['mean_discount_rate_%'] = (cluster_emp['mean_discount_rate'] * 100).round(2)
cluster_emp['median_discount_rate_%'] = (cluster_emp['median_discount_rate'] * 100).round(2)
cluster_emp['mean_applied_discount_pct_%'] = (cluster_emp['mean_applied_discount_pct'] * 100).round(2)
if 'boot_mean_discount_rate' in cluster_emp.columns:
    cluster_emp['boot_mean_discount_rate_%'] = (cluster_emp['boot_mean_discount_rate'] * 100).round(2)
    cluster_emp['ci_low_%'] = (cluster_emp['ci_low'] * 100).round(2)
    cluster_emp['ci_high_%'] = (cluster_emp['ci_high'] * 100).round(2)

# 출력
print("클러스터별 경험적 평균 할인율 (discount_rate) — 관측치 기반")
display_cols = [
    'Cluster', 'n_seqn', 'mean_discount_rate_%', 'median_discount_rate_%',
    'mean_applied_discount_pct_%'
]
if 'boot_mean_discount_rate_%' in cluster_emp.columns:
    display_cols += ['boot_mean_discount_rate_%', 'ci_low_%', 'ci_high_%']

display(cluster_emp[display_cols].sort_values('Cluster').reset_index(drop=True))

# 전체 요약(참고)
overall_mean = float(compare_df['discount_rate'].mean())
print(f"전체 관측 평균 할인율: {overall_mean:.2%}")


클러스터별 경험적 평균 할인율 (discount_rate) — 관측치 기반


,Cluster,n_seqn,mean_discount_rate_%,median_discount_rate_%,mean_applied_discount_pct_%,boot_mean_discount_rate_%,ci_low_%,ci_high_%
0,1,65,7.25,6.46,7.25,7.25,6.83,7.69
1,2,26,9.94,7.25,9.94,9.94,9.34,10.54
2,3,9,12.26,8.96,12.26,12.26,11.40,13.17


전체 관측 평균 할인율: 8.40%


In [9]:
# 진단: 할인 파라미터 및 compare_df 할인 분포 확인
import numpy as np
print('discount_L, discount_k, discount_E0 =', discount_L, discount_k, discount_E0)
print('effort_threshold(clipped) =', float(np.clip(discount_E0, 0.0, 299.999)))

if 'compare_df' in globals():
    print('\ncompare_df 요약:')
    display(compare_df[['monthly_avg_total_score', 'base_premium_P0']].describe())

    if 'discount_rate' in compare_df.columns:
        print('\ndiscount_rate 요약:')
        display(compare_df['discount_rate'].describe())
        nonzero_frac = float((compare_df['discount_rate'] > 0).mean())
        print(f"discount_rate > 0 비율: {nonzero_frac:.3%}")
        print('min,max discount_rate (%):', compare_df['discount_rate'].min()*100, compare_df['discount_rate'].max()*100)
        display(compare_df.loc[:, ['monthly_avg_total_score','effort_score_E','discount_rate','applied_discount_amount','applied_discount_pct','base_premium_P0']].head(20))
    else:
        print('compare_df에 `discount_rate` 컬럼이 없습니다.')
else:
    print('`compare_df`가 존재하지 않습니다. 먼저 보험료 비교 셀을 실행하세요.')

discount_L, discount_k, discount_E0 = 0.20029449495355373 0.04831870607399064 165.04583197569357
effort_threshold(clipped) = 165.04583197569357

compare_df 요약:


,monthly_avg_total_score,base_premium_P0
count,500000.000000,500000.000000
mean,163.162259,62216.900000
std,35.342716,31918.285119
min,101.333333,39380.000000
25%,144.000000,39380.000000
50%,151.000000,39380.000000
75%,159.666667,97580.000000
max,276.000000,124990.000000



discount_rate 요약:


count    500000.000000
mean          0.084008
std           0.050032
min           0.008813
25%           0.053204
50%           0.067410
75%           0.087205
max           0.199358
Name: discount_rate, dtype: float64

discount_rate > 0 비율: 100.000%
min,max discount_rate (%): 0.8813486760025663 19.935841794570734


,monthly_avg_total_score,effort_score_E,discount_rate,applied_discount_amount,applied_discount_pct,base_premium_P0
0,145.000000,145.000000,0.055113,2170.366242,0.055113,39380
1,147.666667,147.666667,0.060407,2378.818887,0.060407,39380
2,143.000000,143.000000,0.051338,2021.685913,0.051338,39380
3,139.333333,139.333333,0.044870,1766.975069,0.044870,39380
4,148.000000,148.000000,0.061088,2405.662851,0.061088,39380
5,135.333333,135.333333,0.038500,1516.129082,0.038500,39380
6,130.000000,130.000000,0.031112,3888.738587,0.031112,124990
7,129.333333,129.333333,0.030275,1192.238701,0.030275,39380
8,139.000000,139.000000,0.044312,1744.989395,0.044312,39380
9,146.666667,146.666667,0.058388,2299.320220,0.058388,39380


In [10]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# ================================================================
# [MASTER REPORT PREP] 마지막 셀 출력에 필요한 계산을 사전 준비
# ================================================================

REPORT_PERCENTILES = [0.05, 0.25, 0.5, 0.75, 0.95]


def _first_available(names, default=np.nan):
    for name in names:
        if name in globals():
            return globals()[name]
    return default


def _safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def _get_pricing_df():
    if 'pricing_df' in globals() and isinstance(pricing_df, pd.DataFrame) and not pricing_df.empty:
        return pricing_df.copy()
    if 'seqn_premium' in globals() and isinstance(seqn_premium, pd.DataFrame) and not seqn_premium.empty:
        return seqn_premium.copy()
    raise ValueError("`pricing_df` 또는 `seqn_premium` 데이터가 없습니다. 시뮬레이션 셀을 먼저 실행하세요.")


def _build_lambda0_by_cluster():
    if 'lambda0_by_cluster' in globals() and isinstance(lambda0_by_cluster, dict):
        return lambda0_by_cluster

    required = [
        'alpha_1', 'alpha_2', 'alpha_3', 'alpha_4',
        'beta_1', 'beta_2', 'beta_3', 'beta_4',
        'lambda1_1', 'lambda1_2', 'lambda1_3', 'lambda1_4',
        'lambda2_1', 'lambda2_2', 'lambda2_3', 'lambda2_4',
        'lambda3_1', 'lambda3_2', 'lambda3_3', 'lambda3_4',
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise ValueError(f"lambda0 계산에 필요한 파라미터가 없습니다: {missing}")

    alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
    beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
    prevalence_by_cluster = {
        1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
        2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
        3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
    }

    out = {}
    for cluster, prev_by_disease in prevalence_by_cluster.items():
        rates = [
            (prev_by_disease[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
            for idx in alpha_by_disease
        ]
        out[cluster] = float(np.sum(rates))
    return out


def _describe_series(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.empty:
        return {
            'count': 0,
            'mean': np.nan,
            'std': np.nan,
            'min': np.nan,
            'p05': np.nan,
            'p25': np.nan,
            'p50': np.nan,
            'p75': np.nan,
            'p95': np.nan,
            'max': np.nan,
        }

    q = s.quantile(REPORT_PERCENTILES)
    return {
        'count': int(s.shape[0]),
        'mean': float(s.mean()),
        'std': float(s.std(ddof=1)) if s.shape[0] > 1 else 0.0,
        'min': float(s.min()),
        'p05': float(q.loc[0.05]),
        'p25': float(q.loc[0.25]),
        'p50': float(q.loc[0.50]),
        'p75': float(q.loc[0.75]),
        'p95': float(q.loc[0.95]),
        'max': float(s.max()),
    }


pricing_for_report = _get_pricing_df()

# 필수 컬럼 보정(호환)
if 'effort_score_E' not in pricing_for_report.columns and 'monthly_avg_total_score' in pricing_for_report.columns:
    pricing_for_report['effort_score_E'] = pricing_for_report['monthly_avg_total_score'].clip(lower=0, upper=300)
if 'monthly_avg_total_score' not in pricing_for_report.columns and 'effort_score_E' in pricing_for_report.columns:
    pricing_for_report['monthly_avg_total_score'] = pricing_for_report['effort_score_E']

if 'expected_loss_E_L' not in pricing_for_report.columns and {'lambda_effective', 'severity'} <= set(list(globals().keys()) + list(pricing_for_report.columns)):
    sev_tmp = _safe_float(_first_available(['severity'], 10000000.0), 10000000.0)
    pricing_for_report['expected_loss_E_L'] = pricing_for_report['lambda_effective'] * sev_tmp

if 'base_premium_P0' in pricing_for_report.columns and 'premium_P' in pricing_for_report.columns:
    pricing_for_report['discount_amount'] = pricing_for_report['base_premium_P0'] - pricing_for_report['premium_P']
else:
    pricing_for_report['discount_amount'] = np.nan

if {'lambda0', 'lambda_effective'} <= set(pricing_for_report.columns):
    pricing_for_report['risk_reduction_amount'] = (pricing_for_report['lambda0'] - pricing_for_report['lambda_effective']) * _safe_float(_first_available(['severity'], 10000000.0), 10000000.0)
else:
    pricing_for_report['risk_reduction_amount'] = np.nan

# ----------------------------------------------------------------
# A) 설정/파라미터/적합도
# ----------------------------------------------------------------
sim_config = {
    'N_BOOT': int(_first_available(['N_BOOT'], pricing_for_report['SEQN'].nunique() if 'SEQN' in pricing_for_report.columns else np.nan))
}

if 'scored_df' in globals() and isinstance(scored_df, pd.DataFrame) and not scored_df.empty:
    if 'bootstrap_id' in scored_df.columns:
        sim_config['N_BOOT'] = int(scored_df['bootstrap_id'].nunique())
    if {'bootstrap_id', 'SEQN'} <= set(scored_df.columns):
        strata_by_boot = scored_df.groupby('bootstrap_id')['SEQN'].nunique()
        sim_config['total_strata_n'] = int(strata_by_boot.median())
    else:
        sim_config['total_strata_n'] = int(scored_df['SEQN'].nunique()) if 'SEQN' in scored_df.columns else np.nan

    if 'sample_idx' in scored_df.columns:
        sim_config['N_SAMPLES_PER_SEQN'] = int(scored_df['sample_idx'].nunique())
    else:
        sim_config['N_SAMPLES_PER_SEQN'] = int(_first_available(['N_SAMPLES_PER_SEQN'], 30))
else:
    sim_config['total_strata_n'] = int(_first_available(['total_strata_n'], np.nan)) if not pd.isna(_first_available(['total_strata_n'], np.nan)) else np.nan
    sim_config['N_SAMPLES_PER_SEQN'] = int(_first_available(['N_SAMPLES_PER_SEQN'], 30))

sim_config['RANDOM_SEED'] = int(_first_available(['RANDOM_SEED'], 42))

L_used_report = _safe_float(_first_available(['L_used', 'L_opt', 'discount_L'], np.nan))
k_used_report = _safe_float(_first_available(['k_used', 'k_opt', 'discount_k'], np.nan))
E0_used_report = _safe_float(_first_available(['E0_used', 'E0_opt', 'discount_E0'], np.nan))
theta_report = _safe_float(_first_available(['theta'], np.mean([
    _safe_float(_first_available(['theta_step'], 5.0), 5.0),
    _safe_float(_first_available(['theta_sleep'], 5.0), 5.0),
    _safe_float(_first_available(['theta_sleep_pattern'], 5.0), 5.0),
])), 5.0)

severity_report = _safe_float(_first_available(['severity'], 10000000.0), 10000000.0)
operating_cost_report = _safe_float(_first_available(['operating_cost'], 0.0), 0.0)

base_premium_map_report = {
    'P1': _safe_float(_first_available(['P1'], np.nan)),
    'p2': _safe_float(_first_available(['p2'], np.nan)),
    'p3': _safe_float(_first_available(['p3'], np.nan)),
}

lambda0_by_cluster_report = _build_lambda0_by_cluster()

mle_summary = {
    'k_opt': _safe_float(_first_available(['k_opt'], np.nan)),
    'E0_opt': _safe_float(_first_available(['E0_opt'], np.nan)),
    'L_opt': _safe_float(_first_available(['L_opt'], np.nan)),
    'NLL': _safe_float(_first_available(['nll_opt'], np.nan)),
    'AIC': _safe_float(_first_available(['aic'], np.nan)),
    'KS_distance': _safe_float(_first_available(['ks_distance'], np.nan)),
}

# ----------------------------------------------------------------
# B) 결과 요약/클러스터/분포/정책효과
# ----------------------------------------------------------------
portfolio_summary = {
    '총 분석 고유 개인 수': int(pricing_for_report.shape[0]),
    '평균 노력 점수': float(pd.to_numeric(pricing_for_report['monthly_avg_total_score'], errors='coerce').mean()) if 'monthly_avg_total_score' in pricing_for_report.columns else np.nan,
    '평균 할인율': float(pd.to_numeric(pricing_for_report['discount_rate'], errors='coerce').mean()) if 'discount_rate' in pricing_for_report.columns else np.nan,
    '평균 최종 보험료': float(pd.to_numeric(pricing_for_report['premium_P'], errors='coerce').mean()) if 'premium_P' in pricing_for_report.columns else np.nan,
    '평균 예상 손실': float(pd.to_numeric(pricing_for_report['expected_loss_E_L'], errors='coerce').mean()) if 'expected_loss_E_L' in pricing_for_report.columns else np.nan,
    '전체 총 예상 이익': float(pd.to_numeric(pricing_for_report['profit_Pi'], errors='coerce').sum()) if 'profit_Pi' in pricing_for_report.columns else np.nan,
    '평균 예상 이익': float(pd.to_numeric(pricing_for_report['profit_Pi'], errors='coerce').mean()) if 'profit_Pi' in pricing_for_report.columns else np.nan,
}

if 'Cluster' in pricing_for_report.columns:
    cluster_detail = (
        pricing_for_report
        .groupby('Cluster', as_index=False)
        .agg(
            개인수=('SEQN', 'count') if 'SEQN' in pricing_for_report.columns else ('Cluster', 'count'),
            평균_노력점수=('monthly_avg_total_score', 'mean') if 'monthly_avg_total_score' in pricing_for_report.columns else ('effort_score_E', 'mean'),
            평균_할인율=('discount_rate', 'mean') if 'discount_rate' in pricing_for_report.columns else ('Cluster', 'size'),
            평균_최종보험료=('premium_P', 'mean') if 'premium_P' in pricing_for_report.columns else ('Cluster', 'size'),
            평균_예상이익=('profit_Pi', 'mean') if 'profit_Pi' in pricing_for_report.columns else ('Cluster', 'size'),
            총_예상이익=('profit_Pi', 'sum') if 'profit_Pi' in pricing_for_report.columns else ('Cluster', 'size'),
        )
    )
    total_profit = cluster_detail['총_예상이익'].sum() if '총_예상이익' in cluster_detail.columns else np.nan
    if pd.notna(total_profit) and total_profit != 0:
        cluster_detail['이익기여도_%'] = cluster_detail['총_예상이익'] / total_profit * 100.0
    else:
        cluster_detail['이익기여도_%'] = np.nan
else:
    cluster_detail = pd.DataFrame()

dist_targets = [
    ('monthly_avg_total_score', '노력점수'),
    ('discount_rate', '할인율'),
    ('profit_Pi', '예상이익'),
]

overall_dist_rows = []
for col, label in dist_targets:
    if col not in pricing_for_report.columns:
        continue
    row = _describe_series(pricing_for_report[col])
    row['지표'] = label
    row['범위'] = '전체'
    overall_dist_rows.append(row)
overall_distribution = pd.DataFrame(overall_dist_rows)

cluster_dist_rows = []
if 'Cluster' in pricing_for_report.columns:
    for cluster_value, sub_df in pricing_for_report.groupby('Cluster'):
        for col, label in dist_targets:
            if col not in sub_df.columns:
                continue
            row = _describe_series(sub_df[col])
            row['지표'] = label
            row['범위'] = f'Cluster {cluster_value}'
            cluster_dist_rows.append(row)
cluster_distribution = pd.DataFrame(cluster_dist_rows)

if 'discount_rate' in pricing_for_report.columns:
    discount_applied_ratio = float((pd.to_numeric(pricing_for_report['discount_rate'], errors='coerce') > 0).mean())
else:
    discount_applied_ratio = np.nan

avg_discount_amount = float(pd.to_numeric(pricing_for_report['discount_amount'], errors='coerce').mean()) if 'discount_amount' in pricing_for_report.columns else np.nan
avg_risk_reduction_amount = float(pd.to_numeric(pricing_for_report['risk_reduction_amount'], errors='coerce').mean()) if 'risk_reduction_amount' in pricing_for_report.columns else np.nan

corr_effort_discount = np.nan
corr_effort_profit = np.nan
if {'monthly_avg_total_score', 'discount_rate'} <= set(pricing_for_report.columns):
    corr_effort_discount = float(pricing_for_report[['monthly_avg_total_score', 'discount_rate']].corr().iloc[0, 1])
if {'monthly_avg_total_score', 'profit_Pi'} <= set(pricing_for_report.columns):
    corr_effort_profit = float(pricing_for_report[['monthly_avg_total_score', 'profit_Pi']].corr().iloc[0, 1])

policy_effect_summary = pd.DataFrame([
    {
        '할인 적용 고객 비율': discount_applied_ratio,
        '평균 할인액(원)': avg_discount_amount,
        '평균 리스크 절감액(원)': avg_risk_reduction_amount,
        '노력점수-할인율 상관계수': corr_effort_discount,
        '노력점수-예상이익 상관계수': corr_effort_profit,
    }
])

# ----------------------------------------------------------------
# C-2) 고급 부트스트랩 불확실성(추가 구현)
# ----------------------------------------------------------------
bootstrap_portfolio_stats = pd.DataFrame()
bootstrap_uncertainty_summary = pd.DataFrame()
seqn_profit_boot_ci = pd.DataFrame()

can_bootstrap = (
    'scored_df' in globals()
    and isinstance(scored_df, pd.DataFrame)
    and {'bootstrap_id', 'sample_idx', 'SEQN', 'Cluster', 'total_score'} <= set(scored_df.columns)
)

if can_bootstrap:
    base_map = {
        1: _safe_float(_first_available(['P1'], np.nan)),
        2: _safe_float(_first_available(['p2'], np.nan)),
        3: _safe_float(_first_available(['p3'], np.nan)),
    }

    monthly_boot = (
        scored_df[scored_df['sample_idx'].between(1, 30)]
        .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
        .agg(
            monthly_avg_total_score=('total_score', 'mean'),
            observed_days=('sample_idx', 'nunique')
        )
    )
    monthly_boot = monthly_boot[monthly_boot['observed_days'] == 30].copy()

    if not monthly_boot.empty:
        monthly_boot['base_premium_P0'] = monthly_boot['Cluster'].map(base_map)
        monthly_boot['lambda0'] = monthly_boot['Cluster'].map(lambda0_by_cluster_report)

        if monthly_boot[['base_premium_P0', 'lambda0']].isna().any().any():
            raise ValueError("부트스트랩 불확실성 계산 중 Cluster 파라미터 매핑 누락이 있습니다.")

        monthly_boot['effort_score_E'] = monthly_boot['monthly_avg_total_score'].clip(lower=0, upper=300)
        monthly_boot['discount_rate'] = np.clip(
            L_used_report / (1.0 + np.exp(-k_used_report * (monthly_boot['effort_score_E'] - E0_used_report))),
            0.0,
            1.0,
        )
        monthly_boot['raw_discount_amount'] = monthly_boot['base_premium_P0'] * monthly_boot['discount_rate']

        max_discount_amt = _safe_float(_first_available(['max_discount_amount'], 0.0), 0.0)
        if max_discount_amt > 0:
            monthly_boot['applied_discount_amount'] = monthly_boot['raw_discount_amount'].clip(upper=max_discount_amt)
        else:
            monthly_boot['applied_discount_amount'] = monthly_boot['raw_discount_amount']

        monthly_boot['premium_P'] = monthly_boot['base_premium_P0'] - monthly_boot['applied_discount_amount']
        monthly_boot['delta_E'] = np.maximum(0.0, monthly_boot['effort_score_E'] - E0_used_report) / 300.0
        monthly_boot['lambda_effective'] = monthly_boot['lambda0'] * np.exp(-theta_report * monthly_boot['delta_E'])
        monthly_boot['expected_loss_E_L'] = monthly_boot['lambda_effective'] * severity_report
        monthly_boot['profit_Pi'] = monthly_boot['premium_P'] - monthly_boot['expected_loss_E_L'] - operating_cost_report

        bootstrap_portfolio_stats = (
            monthly_boot
            .groupby('bootstrap_id', as_index=False)
            .agg(
                n_seqn=('SEQN', 'nunique'),
                mean_profit=('profit_Pi', 'mean'),
                mean_discount_rate=('discount_rate', 'mean'),
                mean_premium=('premium_P', 'mean'),
                mean_expected_loss=('expected_loss_E_L', 'mean'),
                total_profit=('profit_Pi', 'sum'),
            )
        )

        if not bootstrap_portfolio_stats.empty:
            ci_mean_profit = np.percentile(bootstrap_portfolio_stats['mean_profit'], [2.5, 97.5])
            ci_mean_discount = np.percentile(bootstrap_portfolio_stats['mean_discount_rate'], [2.5, 97.5])
            ci_total_profit = np.percentile(bootstrap_portfolio_stats['total_profit'], [2.5, 97.5])

            bootstrap_uncertainty_summary = pd.DataFrame([
                {
                    '지표': '부트스트랩별 포트폴리오 평균 이익',
                    '평균': float(bootstrap_portfolio_stats['mean_profit'].mean()),
                    '표준편차': float(bootstrap_portfolio_stats['mean_profit'].std(ddof=1)),
                    '95% CI 하한': float(ci_mean_profit[0]),
                    '95% CI 상한': float(ci_mean_profit[1]),
                },
                {
                    '지표': '부트스트랩별 포트폴리오 평균 할인율',
                    '평균': float(bootstrap_portfolio_stats['mean_discount_rate'].mean()),
                    '표준편차': float(bootstrap_portfolio_stats['mean_discount_rate'].std(ddof=1)),
                    '95% CI 하한': float(ci_mean_discount[0]),
                    '95% CI 상한': float(ci_mean_discount[1]),
                },
                {
                    '지표': '부트스트랩별 포트폴리오 총 이익',
                    '평균': float(bootstrap_portfolio_stats['total_profit'].mean()),
                    '표준편차': float(bootstrap_portfolio_stats['total_profit'].std(ddof=1)),
                    '95% CI 하한': float(ci_total_profit[0]),
                    '95% CI 상한': float(ci_total_profit[1]),
                },
            ])

            seqn_profit_boot_ci = (
                monthly_boot
                .groupby('SEQN')['profit_Pi']
                .agg(
                    boot_mean='mean',
                    boot_std='std',
                    q025=lambda x: np.quantile(x, 0.025),
                    q975=lambda x: np.quantile(x, 0.975),
                    n_boot='count',
                )
                .reset_index()
                .sort_values('boot_std', ascending=False)
                .reset_index(drop=True)
            )

master_report_context = {
    'sim_config': sim_config,
    'applied_params': {
        'L_used': L_used_report,
        'k_used': k_used_report,
        'E0_used': E0_used_report,
        'theta': theta_report,
        'severity': severity_report,
        'operating_cost': operating_cost_report,
        **base_premium_map_report,
    },
    'disease_params': {
        'lambdas': {
            'cluster1': [
                _safe_float(_first_available(['lambda1_1'], np.nan)),
                _safe_float(_first_available(['lambda1_2'], np.nan)),
                _safe_float(_first_available(['lambda1_3'], np.nan)),
                _safe_float(_first_available(['lambda1_4'], np.nan)),
            ],
            'cluster2': [
                _safe_float(_first_available(['lambda2_1'], np.nan)),
                _safe_float(_first_available(['lambda2_2'], np.nan)),
                _safe_float(_first_available(['lambda2_3'], np.nan)),
                _safe_float(_first_available(['lambda2_4'], np.nan)),
            ],
            'cluster3': [
                _safe_float(_first_available(['lambda3_1'], np.nan)),
                _safe_float(_first_available(['lambda3_2'], np.nan)),
                _safe_float(_first_available(['lambda3_3'], np.nan)),
                _safe_float(_first_available(['lambda3_4'], np.nan)),
            ],
        },
        'alphas': [
            _safe_float(_first_available(['alpha_1'], np.nan)),
            _safe_float(_first_available(['alpha_2'], np.nan)),
            _safe_float(_first_available(['alpha_3'], np.nan)),
            _safe_float(_first_available(['alpha_4'], np.nan)),
        ],
        'betas': [
            _safe_float(_first_available(['beta_1'], np.nan)),
            _safe_float(_first_available(['beta_2'], np.nan)),
            _safe_float(_first_available(['beta_3'], np.nan)),
            _safe_float(_first_available(['beta_4'], np.nan)),
        ],
        'lambda0_by_cluster': lambda0_by_cluster_report,
    },
    'mle_summary': mle_summary,
    'portfolio_summary': portfolio_summary,
    'cluster_detail': cluster_detail,
    'overall_distribution': overall_distribution,
    'cluster_distribution': cluster_distribution,
    'policy_effect_summary': policy_effect_summary,
    'bootstrap_portfolio_stats': bootstrap_portfolio_stats,
    'bootstrap_uncertainty_summary': bootstrap_uncertainty_summary,
    'seqn_profit_boot_ci': seqn_profit_boot_ci,
}

print("master_report_context 계산 완료: 마지막 셀에서 전체 리포트를 출력할 수 있습니다.")

master_report_context 계산 완료: 마지막 셀에서 전체 리포트를 출력할 수 있습니다.


In [11]:
# import pandas as pd
# import numpy as np
# from IPython.display import display, Markdown

# # =================================================================
# # [FINAL MASTER REPORT]
# # =================================================================

# if 'master_report_context' not in globals():
#     raise ValueError("`master_report_context`가 없습니다. 바로 위의 [MASTER REPORT PREP] 셀을 먼저 실행하세요.")

# ctx = master_report_context

# sim_config = pd.DataFrame([ctx['sim_config']])
# applied_params = pd.DataFrame([ctx['applied_params']])
# mle_summary_df = pd.DataFrame([ctx['mle_summary']])
# portfolio_summary_df = pd.DataFrame([ctx['portfolio_summary']])
# cluster_detail_df = ctx['cluster_detail']
# overall_dist_df = ctx['overall_distribution']
# cluster_dist_df = ctx['cluster_distribution']
# policy_effect_df = ctx['policy_effect_summary']
# bootstrap_portfolio_stats_df = ctx['bootstrap_portfolio_stats']
# bootstrap_uncertainty_df = ctx['bootstrap_uncertainty_summary']
# seqn_profit_boot_ci_df = ctx['seqn_profit_boot_ci']

# disease_params = ctx['disease_params']
# lambda_df = pd.DataFrame([
#     {
#         'Cluster': 1,
#         'lambda1_1': disease_params['lambdas']['cluster1'][0],
#         'lambda1_2': disease_params['lambdas']['cluster1'][1],
#         'lambda1_3': disease_params['lambdas']['cluster1'][2],
#         'lambda1_4': disease_params['lambdas']['cluster1'][3],
#     },
#     {
#         'Cluster': 2,
#         'lambda2_1': disease_params['lambdas']['cluster2'][0],
#         'lambda2_2': disease_params['lambdas']['cluster2'][1],
#         'lambda2_3': disease_params['lambdas']['cluster2'][2],
#         'lambda2_4': disease_params['lambdas']['cluster2'][3],
#     },
#     {
#         'Cluster': 3,
#         'lambda3_1': disease_params['lambdas']['cluster3'][0],
#         'lambda3_2': disease_params['lambdas']['cluster3'][1],
#         'lambda3_3': disease_params['lambdas']['cluster3'][2],
#         'lambda3_4': disease_params['lambdas']['cluster3'][3],
#     },
# ])

# alphas_betas_df = pd.DataFrame([
#     {
#         'alpha_1': disease_params['alphas'][0],
#         'alpha_2': disease_params['alphas'][1],
#         'alpha_3': disease_params['alphas'][2],
#         'alpha_4': disease_params['alphas'][3],
#         'beta_1': disease_params['betas'][0],
#         'beta_2': disease_params['betas'][1],
#         'beta_3': disease_params['betas'][2],
#         'beta_4': disease_params['betas'][3],
#     }
# ])

# lambda0_cluster_df = (
#     pd.DataFrame([
#         {'Cluster': k, 'lambda0': v}
#         for k, v in disease_params['lambda0_by_cluster'].items()
#     ])
#     .sort_values('Cluster')
#     .reset_index(drop=True)
# )

# # 보기 형식 보정
# for c in ['평균 할인율', '평균 노력 점수']:
#     if c in portfolio_summary_df.columns:
#         portfolio_summary_df[c] = portfolio_summary_df[c].astype(float)

# if not cluster_detail_df.empty:
#     for col in ['평균_노력점수', '평균_할인율', '평균_최종보험료', '평균_예상이익', '총_예상이익', '이익기여도_%']:
#         if col in cluster_detail_df.columns:
#             cluster_detail_df[col] = pd.to_numeric(cluster_detail_df[col], errors='coerce')

# if not overall_dist_df.empty:
#     overall_dist_df = overall_dist_df[['범위', '지표', 'count', 'mean', 'std', 'min', 'p05', 'p25', 'p50', 'p75', 'p95', 'max']]
# if not cluster_dist_df.empty:
#     cluster_dist_df = cluster_dist_df[['범위', '지표', 'count', 'mean', 'std', 'min', 'p05', 'p25', 'p50', 'p75', 'p95', 'max']]

# display(Markdown("# 📊 보험 시뮬레이션 결과 마스터 리포트"))

# display(Markdown("## A. 시뮬레이션 설정 및 모델 파라미터"))

# display(Markdown("### A-1) 시뮬레이션 기본 구성"))
# display(sim_config)

# display(Markdown("### A-2) 정책/모델 입력 파라미터 (최종 적용값)"))
# display(applied_params)
# display(Markdown("#### 질병 발생률 관련 파라미터 (lambdas)"))

# # --- lambda_df 통일 출력 (cluster-specific 컬럼명으로 인해 NaN이 보이는 문제 보정) ---
# def _normalize_lambda_df(df):
#     # 이미 통일된 형식이면 그대로 반환
#     if {'lambda_1','lambda_2','lambda_3','lambda_4'}.issubset(df.columns):
#         return df[['Cluster','lambda_1','lambda_2','lambda_3','lambda_4']]

#     rows = []
#     for _, r in df.iterrows():
#         cluster = int(r['Cluster']) if not pd.isna(r['Cluster']) else None
#         if cluster == 1:
#             vals = [r.get('lambda1_1'), r.get('lambda1_2'), r.get('lambda1_3'), r.get('lambda1_4')]
#         elif cluster == 2:
#             vals = [r.get('lambda2_1'), r.get('lambda2_2'), r.get('lambda2_3'), r.get('lambda2_4')]
#         elif cluster == 3:
#             vals = [r.get('lambda3_1'), r.get('lambda3_2'), r.get('lambda3_3'), r.get('lambda3_4')]
#         else:
#             vals = [r.get('lambda1_1') or r.get('lambda2_1') or r.get('lambda3_1'),
#                     r.get('lambda1_2') or r.get('lambda2_2') or r.get('lambda3_2'),
#                     r.get('lambda1_3') or r.get('lambda2_3') or r.get('lambda3_3'),
#                     r.get('lambda1_4') or r.get('lambda2_4') or r.get('lambda3_4')]
#         rows.append({'Cluster': cluster, 'lambda_1': vals[0], 'lambda_2': vals[1], 'lambda_3': vals[2], 'lambda_4': vals[3]})
#     return pd.DataFrame(rows)

# display(_normalize_lambda_df(lambda_df))
# display(Markdown("#### 질병 발생률 관련 파라미터 (alphas, betas)"))
# display(alphas_betas_df)
# display(Markdown("#### 클러스터별 기준 발생률 (lambda0)"))
# display(lambda0_cluster_df)

# display(Markdown("### A-3) 노력 점수 분포 MLE 결과 요약"))
# display(mle_summary_df)

# display(Markdown("## B. 시뮬레이션 결과 요약 및 핵심 지표"))

# display(Markdown("### B-1) 전체 포트폴리오 요약 통계"))
# display(portfolio_summary_df)

# display(Markdown("### B-2) 클러스터(위험군)별 상세 분석"))
# if cluster_detail_df.empty:
#     print("클러스터 상세 분석 데이터가 없습니다.")
# else:
#     display(cluster_detail_df.sort_values('Cluster').reset_index(drop=True))

# display(Markdown("### B-3) 주요 지표 분포 분석 (전체)"))
# if overall_dist_df.empty:
#     print("전체 분포 통계 데이터가 없습니다.")
# else:
#     display(overall_dist_df.reset_index(drop=True))

# display(Markdown("### B-3) 주요 지표 분포 분석 (클러스터별)"))
# if cluster_dist_df.empty:
#     print("클러스터별 분포 통계 데이터가 없습니다.")
# else:
#     display(cluster_dist_df.sort_values(['범위', '지표']).reset_index(drop=True))

# display(Markdown("### B-4) 정책 효과 및 가치 제안"))
# display(policy_effect_df)

# display(Markdown("""
# - **할인 적용 고객 비율**이 높을수록 행동 기반 인센티브가 실제로 작동하고 있음을 의미합니다.
# - **평균 할인액**은 고객 체감 혜택의 크기를 보여주며, 상품 매력도 개선 근거가 됩니다.
# - **노력점수-할인율/이익 상관**은 "노력의 가치"를 정량적으로 설명합니다.
# - **평균 리스크 절감액**은 행동 변화가 예상 손실 감소로 이어지는 위험관리 효과를 보여줍니다.
# """))

# display(Markdown("## C. 부트스트래핑 활용 목적 및 추가 분석"))

# display(Markdown("### C-1) 현재 코드의 부트스트래핑 활용 목적"))
# display(Markdown("""
# 1. 5000회 부트스트랩은 개인별 30일 행동 패턴의 변동성을 모사하기 위한 절차입니다.
# 2. 부트스트랩 결과를 평균해 `monthly_avg_total_score`를 안정적인 기대 행동 점수로 추정합니다.
# 3. 이를 통해 일시적 변동(노이즈)에 덜 민감한 보험료 산정 기준을 제공합니다.
# """))

# display(Markdown("### C-2) 고급 부트스트랩 불확실성 추정 (추가 구현)"))
# if bootstrap_uncertainty_df.empty:
#     print("부트스트랩 불확실성 요약을 계산하지 못했습니다. `scored_df`와 필수 파라미터를 확인하세요.")
# else:
#     display(bootstrap_uncertainty_df)

#     display(Markdown("#### 부트스트랩별 포트폴리오 통계 (일부)"))
#     display(bootstrap_portfolio_stats_df.head(20))

#     display(Markdown("#### 개인별 profit 부트스트랩 변동성 상위 20명"))
#     if seqn_profit_boot_ci_df.empty:
#         print("개인별 부트스트랩 변동성 데이터가 없습니다.")
#     else:
#         display(seqn_profit_boot_ci_df.head(20))

# print("리포트 출력 완료")

### 리포트

In [12]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from scipy import stats

# =================================================================
# [FINAL MASTER REPORT] 
# =================================================================

if 'master_report_context' not in globals():
    raise ValueError("`master_report_context`가 없습니다. 바로 위의 [MASTER REPORT PREP] 셀을 먼저 실행하세요.")

ctx = master_report_context
full_report_markdown = ""  # 여기에 모든 마크다운 내용을 축적합니다.

def _safe_df(value):
    return value.copy() if isinstance(value, pd.DataFrame) else pd.DataFrame()

def _safe_paired_tests(with_score_series, cluster_series):
    a = pd.to_numeric(with_score_series, errors='coerce')
    b = pd.to_numeric(cluster_series, errors='coerce')
    valid = a.notna() & b.notna()
    a = a[valid]
    b = b[valid]
    if len(a) < 2:
        return np.nan, np.nan, np.nan, np.nan

    t_res = stats.ttest_rel(a, b)

    try:
        w_res = stats.wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
        w_stat, w_p = float(w_res.statistic), float(w_res.pvalue)
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    return float(t_res.statistic), float(t_res.pvalue), w_stat, w_p

def _build_insurer_pl_from_bootstrap(premium_boot, spend_boot):
    required_premium_cols = {'bootstrap_id', 'mean_premium_cluster_only', 'mean_premium_with_score'}
    required_spend_cols = {'bootstrap_id', 'mean_exp_cluster_only', 'mean_exp_with_score'}

    if not required_premium_cols.issubset(premium_boot.columns):
        return pd.DataFrame(), pd.DataFrame()
    if not required_spend_cols.issubset(spend_boot.columns):
        return pd.DataFrame(), pd.DataFrame()

    merged = pd.merge(
        premium_boot[list(required_premium_cols)],
        spend_boot[list(required_spend_cols)],
        on='bootstrap_id',
        how='inner'
    )

    if merged.empty:
        return pd.DataFrame(), pd.DataFrame()

    merged['mean_profit_cluster_only'] = merged['mean_premium_cluster_only'] - merged['mean_exp_cluster_only']
    merged['mean_profit_with_score'] = merged['mean_premium_with_score'] - merged['mean_exp_with_score']
    merged['mean_diff_with_score_minus_cluster'] = merged['mean_profit_with_score'] - merged['mean_profit_cluster_only']

    pl_bootstrap_summary = merged[[
        'bootstrap_id',
        'mean_profit_cluster_only',
        'mean_profit_with_score',
        'mean_diff_with_score_minus_cluster',
    ]].copy()

    diffs = pl_bootstrap_summary['mean_diff_with_score_minus_cluster'].to_numpy(dtype=float)
    rng = np.random.default_rng(42)
    B = 5000
    boot_means = np.empty(B)
    for i in range(B):
        sample = rng.choice(diffs, size=len(diffs), replace=True)
        boot_means[i] = sample.mean()
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

    t_stat, t_p, w_stat, w_p = _safe_paired_tests(
        pl_bootstrap_summary['mean_profit_with_score'],
        pl_bootstrap_summary['mean_profit_cluster_only']
    )

    pl_stats_result = pd.DataFrame([
        {
            'n_bootstrap_id': int(len(pl_bootstrap_summary)),
            'mean_diff(with_score-cluster)': float(np.mean(diffs)),
            'bootstrap95ci_low': float(ci_low),
            'bootstrap95ci_high': float(ci_high),
            'paired_t_stat': t_stat,
            'paired_t_pvalue': t_p,
            'wilcoxon_stat': w_stat,
            'wilcoxon_pvalue': w_p,
        }
    ])

    return pl_bootstrap_summary, pl_stats_result

# --- master_report_context에서 DataFrame 변환 (기존 코드와 동일) ---
sim_config_df = pd.DataFrame([ctx['sim_config']])
applied_params_df = pd.DataFrame([ctx['applied_params']])
mle_summary_df = pd.DataFrame([ctx['mle_summary']])
portfolio_summary_df = pd.DataFrame([ctx['portfolio_summary']])
cluster_detail_df = _safe_df(ctx['cluster_detail'])
overall_dist_df = _safe_df(ctx['overall_distribution'])
cluster_dist_df = _safe_df(ctx['cluster_distribution'])
policy_effect_df = _safe_df(ctx['policy_effect_summary'])
bootstrap_portfolio_stats_df = _safe_df(ctx['bootstrap_portfolio_stats'])
bootstrap_uncertainty_df = _safe_df(ctx['bootstrap_uncertainty_summary'])
seqn_profit_boot_ci_df = _safe_df(ctx['seqn_profit_boot_ci'])

# --- 추가: 모델 비교 통계 (ctx 우선, 없으면 globals fallback) ---
premium_bootstrap_summary_df = _safe_df(ctx.get('premium_bootstrap_summary', pd.DataFrame()))
premium_stats_result_df = _safe_df(ctx.get('premium_stats_result', pd.DataFrame()))
insurer_pl_bootstrap_summary_df = _safe_df(ctx.get('insurer_pl_bootstrap_summary', pd.DataFrame()))
insurer_pl_stats_result_df = _safe_df(ctx.get('insurer_pl_stats_result', pd.DataFrame()))

if premium_bootstrap_summary_df.empty and 'bootstrap_summary' in globals():
    premium_bootstrap_summary_df = _safe_df(globals()['bootstrap_summary'])
if premium_stats_result_df.empty and 'stats_result' in globals():
    premium_stats_result_df = _safe_df(globals()['stats_result'])

if insurer_pl_bootstrap_summary_df.empty or insurer_pl_stats_result_df.empty:
    if 'bootstrap_summary' in globals() and 'spend_bootstrap_summary' in globals():
        pl_summary, pl_stats = _build_insurer_pl_from_bootstrap(
            _safe_df(globals()['bootstrap_summary']),
            _safe_df(globals()['spend_bootstrap_summary'])
        )
        if insurer_pl_bootstrap_summary_df.empty:
            insurer_pl_bootstrap_summary_df = pl_summary
        if insurer_pl_stats_result_df.empty:
            insurer_pl_stats_result_df = pl_stats

disease_params = ctx['disease_params']
lambda_df = pd.DataFrame([
    {
        'Cluster': 1,
        'lambda1_1': disease_params['lambdas']['cluster1'][0],
        'lambda1_2': disease_params['lambdas']['cluster1'][1],
        'lambda1_3': disease_params['lambdas']['cluster1'][2],
        'lambda1_4': disease_params['lambdas']['cluster1'][3],
    },
    {
        'Cluster': 2,
        'lambda2_1': disease_params['lambdas']['cluster2'][0],
        'lambda2_2': disease_params['lambdas']['cluster2'][1],
        'lambda2_3': disease_params['lambdas']['cluster2'][2],
        'lambda2_4': disease_params['lambdas']['cluster2'][3],
    },
    {
        'Cluster': 3,
        'lambda3_1': disease_params['lambdas']['cluster3'][0],
        'lambda3_2': disease_params['lambdas']['cluster3'][1],
        'lambda3_3': disease_params['lambdas']['cluster3'][2],
        'lambda3_4': disease_params['lambdas']['cluster3'][3],
    },
])

def _normalize_lambda_df(df):
    if {'lambda_1', 'lambda_2', 'lambda_3', 'lambda_4'}.issubset(df.columns):
        return df[['Cluster', 'lambda_1', 'lambda_2', 'lambda_3', 'lambda_4']]

    rows = []
    for _, r in df.iterrows():
        cluster = int(r['Cluster']) if not pd.isna(r['Cluster']) else None
        if cluster == 1:
            vals = [r.get('lambda1_1'), r.get('lambda1_2'), r.get('lambda1_3'), r.get('lambda1_4')]
        elif cluster == 2:
            vals = [r.get('lambda2_1'), r.get('lambda2_2'), r.get('lambda2_3'), r.get('lambda2_4')]
        elif cluster == 3:
            vals = [r.get('lambda3_1'), r.get('lambda3_2'), r.get('lambda3_3'), r.get('lambda3_4')]
        else:
            vals = [
                r.get('lambda1_1') or r.get('lambda2_1') or r.get('lambda3_1'),
                r.get('lambda1_2') or r.get('lambda2_2') or r.get('lambda3_2'),
                r.get('lambda1_3') or r.get('lambda2_3') or r.get('lambda3_3'),
                r.get('lambda1_4') or r.get('lambda2_4') or r.get('lambda3_4'),
            ]
        rows.append({
            'Cluster': cluster,
            'lambda_1': vals[0],
            'lambda_2': vals[1],
            'lambda_3': vals[2],
            'lambda_4': vals[3],
        })

    return pd.DataFrame(rows)

lambda_df = _normalize_lambda_df(lambda_df)

alphas_betas_df = pd.DataFrame([
    {
        'alpha_1': disease_params['alphas'][0],
        'alpha_2': disease_params['alphas'][1],
        'alpha_3': disease_params['alphas'][2],
        'alpha_4': disease_params['alphas'][3],
        'beta_1': disease_params['betas'][0],
        'beta_2': disease_params['betas'][1],
        'beta_3': disease_params['betas'][2],
        'beta_4': disease_params['betas'][3],
    }
])

lambda0_cluster_df = (
    pd.DataFrame([
        {'Cluster': k, 'lambda0': v}
        for k, v in disease_params['lambda0_by_cluster'].items()
    ])
    .sort_values('Cluster')
    .reset_index(drop=True)
)

# 보기 형식 보정 (기존 코드와 동일)
for c in ['평균 할인율', '평균 노력 점수']:
    if c in portfolio_summary_df.columns:
        portfolio_summary_df[c] = portfolio_summary_df[c].astype(float)

if not cluster_detail_df.empty:
    for col in ['평균_노력점수', '평균_할인율', '평균_최종보험료', '평균_예상이익', '총_예상이익', '이익기여도_%']:
        if col in cluster_detail_df.columns:
            cluster_detail_df[col] = pd.to_numeric(cluster_detail_df[col], errors='coerce')

if not overall_dist_df.empty:
    overall_dist_df = overall_dist_df[['범위', '지표', 'count', 'mean', 'std', 'min', 'p05', 'p25', 'p50', 'p75', 'p95', 'max']]
if not cluster_dist_df.empty:
    cluster_dist_df = cluster_dist_df[['범위', '지표', 'count', 'mean', 'std', 'min', 'p05', 'p25', 'p50', 'p75', 'p95', 'max']]

# --- 모든 내용을 하나의 마크다운 문자열로 축적 ---

full_report_markdown += "# 📊 보험 시뮬레이션 결과 마스터 리포트\n\n"

full_report_markdown += "## A. 시뮬레이션 설정 및 모델 파라미터\n\n"

full_report_markdown += "### A-1) 시뮬레이션 기본 구성\n"
full_report_markdown += sim_config_df.to_string(index=False) + "\n\n"

full_report_markdown += "### A-2) 정책/모델 입력 파라미터 (최종 적용값)\n"
full_report_markdown += applied_params_df.to_string(index=False) + "\n\n"
full_report_markdown += "#### 질병 발생률 관련 파라미터 (lambdas)\n"
full_report_markdown += lambda_df.to_string(index=False) + "\n\n"
full_report_markdown += "#### 질병 발생률 관련 파라미터 (alphas, betas)\n"
full_report_markdown += alphas_betas_df.to_string(index=False) + "\n\n"
full_report_markdown += "#### 클러스터별 기준 발생률 (lambda0)\n"
full_report_markdown += lambda0_cluster_df.to_string(index=False) + "\n\n"

full_report_markdown += "### A-3) 노력 점수 분포 MLE 결과 요약\n"
full_report_markdown += mle_summary_df.to_string(index=False) + "\n\n"

full_report_markdown += "## B. 시뮬레이션 결과 요약 및 핵심 지표\n\n"

full_report_markdown += "### B-1) 전체 포트폴리오 요약 통계\n"
full_report_markdown += portfolio_summary_df.to_string(index=False) + "\n\n"

full_report_markdown += "### B-2) 클러스터(위험군)별 상세 분석\n"
if cluster_detail_df.empty:
    full_report_markdown += "클러스터 상세 분석 데이터가 없습니다.\n\n"
else:
    full_report_markdown += cluster_detail_df.sort_values('Cluster').reset_index(drop=True).to_string(index=False) + "\n\n"

full_report_markdown += "### B-3) 주요 지표 분포 분석 (전체)\n"
if overall_dist_df.empty:
    full_report_markdown += "전체 분포 통계 데이터가 없습니다.\n\n"
else:
    full_report_markdown += overall_dist_df.reset_index(drop=True).to_string(index=False) + "\n\n"

full_report_markdown += "### B-4) 주요 지표 분포 분석 (클러스터별)\n"
if cluster_dist_df.empty:
    full_report_markdown += "클러스터별 분포 통계 데이터가 없습니다.\n\n"
else:
    full_report_markdown += cluster_dist_df.sort_values(['범위', '지표']).reset_index(drop=True).to_string(index=False) + "\n\n"

full_report_markdown += "### B-5) 정책 효과 및 가치 제안\n"
full_report_markdown += policy_effect_df.to_string(index=False) + "\n\n"
full_report_markdown += """
- **할인 적용 고객 비율**이 높을수록 행동 기반 인센티브가 실제로 작동하고 있음을 의미합니다.
- **평균 할인액**은 고객 체감 혜택의 크기를 보여주며, 상품 매력도 개선 근거가 됩니다.
- **노력점수-할인율/이익 상관**은 \"노력의 가치\"를 정량적으로 설명합니다.
- **평균 리스크 절감액**은 행동 변화가 예상 손실 감소로 이어지는 위험관리 효과를 보여줍니다.
"""
full_report_markdown += "\n\n"

full_report_markdown += "## C. 클러스터링 단독 vs 클러스터링+할인 통계 분석\n\n"

full_report_markdown += "### C-1) 보험료 비교 통계 (bootstrap 단위)\n"
if premium_bootstrap_summary_df.empty:
    full_report_markdown += "보험료 비교 요약 데이터가 없습니다. `보험료 비교` 셀을 먼저 실행하세요.\n\n"
else:
    cols = [
        'bootstrap_id',
        'mean_premium_cluster_only',
        'mean_premium_with_score',
        'mean_diff_score_minus_cluster',
    ]
    cols = [c for c in cols if c in premium_bootstrap_summary_df.columns]
    full_report_markdown += premium_bootstrap_summary_df[cols].head(20).to_string(index=False) + "\n\n"

if premium_stats_result_df.empty:
    full_report_markdown += "보험료 비교 통계 검정 결과가 없습니다.\n\n"
else:
    full_report_markdown += "#### 보험료 비교 검정 결과\n"
    full_report_markdown += premium_stats_result_df.to_string(index=False) + "\n\n"

full_report_markdown += "### C-2) 보험사 손익 비교 통계 (bootstrap 단위)\n"
if insurer_pl_bootstrap_summary_df.empty:
    full_report_markdown += "보험사 손익 비교 요약 데이터가 없습니다. `보험료 비교` 및 `보험사 지출 비교` 셀을 실행하세요.\n\n"
else:
    pl_cols = [
        'bootstrap_id',
        'mean_profit_cluster_only',
        'mean_profit_with_score',
        'mean_diff_with_score_minus_cluster',
    ]
    pl_cols = [c for c in pl_cols if c in insurer_pl_bootstrap_summary_df.columns]
    full_report_markdown += insurer_pl_bootstrap_summary_df[pl_cols].head(20).to_string(index=False) + "\n\n"

if insurer_pl_stats_result_df.empty:
    full_report_markdown += "보험사 손익 비교 통계 검정 결과가 없습니다.\n\n"
else:
    full_report_markdown += "#### 보험사 손익 비교 검정 결과\n"
    full_report_markdown += insurer_pl_stats_result_df.to_string(index=False) + "\n\n"

full_report_markdown += """
- `mean_diff`가 음수이면 클러스터링+할인 모델이 평균 보험료(또는 평균 손익)를 낮춘 방향입니다.
- p-value가 작을수록 두 모델 차이가 우연이 아닐 가능성이 큽니다.
- 95% 신뢰구간에 0이 포함되지 않으면 모델 차이의 방향성이 더 안정적이라고 해석할 수 있습니다.
"""
full_report_markdown += "\n"

full_report_markdown += "## D. 부트스트래핑 활용 목적 및 추가 분석\n\n"

full_report_markdown += "### D-1) 현재 코드의 부트스트래핑 활용 목적\n"
full_report_markdown += """
1. 5000회 부트스트랩은 개인별 30일 행동 패턴의 변동성을 모사하기 위한 절차입니다.
2. 부트스트랩 결과를 평균해 `monthly_avg_total_score`를 안정적인 기대 행동 점수로 추정합니다.
3. 이를 통해 일시적 변동(노이즈)에 덜 민감한 보험료 산정 기준을 제공합니다.
"""
full_report_markdown += "\n"

full_report_markdown += "### D-2) 고급 부트스트랩 불확실성 추정 (추가 구현)\n"
if bootstrap_uncertainty_df.empty:
    full_report_markdown += "부트스트랩 불확실성 요약을 계산하지 못했습니다. `scored_df`와 필수 파라미터를 확인하세요.\n\n"
else:
    full_report_markdown += bootstrap_uncertainty_df.to_string(index=False) + "\n\n"

    full_report_markdown += "#### 부트스트랩별 포트폴리오 통계 (일부)\n"
    full_report_markdown += bootstrap_portfolio_stats_df.head(20).to_string(index=False) + "\n\n"

    full_report_markdown += "#### 개인별 profit 부트스트랩 변동성 상위 20명\n"
    if seqn_profit_boot_ci_df.empty:
        full_report_markdown += "개인별 부트스트랩 변동성 데이터가 없습니다.\n\n"
    else:
        full_report_markdown += seqn_profit_boot_ci_df.head(20).to_string(index=False) + "\n\n"

# 최종 출력
print("리포트 출력 완료")  # 마지막에 이 메시지를 먼저 출력
display(Markdown(full_report_markdown))

리포트 출력 완료


# 📊 보험 시뮬레이션 결과 마스터 리포트

## A. 시뮬레이션 설정 및 모델 파라미터

### A-1) 시뮬레이션 기본 구성
 N_BOOT  total_strata_n  N_SAMPLES_PER_SEQN  RANDOM_SEED
   5000             100                  30           42

### A-2) 정책/모델 입력 파라미터 (최종 적용값)
 L_used  k_used  E0_used  theta   severity  operating_cost      P1      p2       p3
    0.2    0.03    150.0    3.0 10000000.0             0.0 39380.0 97580.0 124990.0

#### 질병 발생률 관련 파라미터 (lambdas)
 Cluster  lambda_1  lambda_2  lambda_3  lambda_4
       1      0.62      0.84      0.59      1.08
       2      2.90      3.25      2.47      3.67
       3      7.39      8.85      5.60      8.24

#### 질병 발생률 관련 파라미터 (alphas, betas)
 alpha_1  alpha_2  alpha_3  alpha_4   beta_1   beta_2  beta_3   beta_4
    2.01      2.3     1.71     2.63 0.000636 0.000086 0.00009 0.000057

#### 클러스터별 기준 발생률 (lambda0)
 Cluster  lambda0
       1 0.000282
       2 0.001248
       3 0.003141

### A-3) 노력 점수 분포 MLE 결과 요약
   k_opt     E0_opt    L_opt          NLL          AIC  KS_distance
0.048319 165.045832 0.200294 37436.847472 74877.694945      0.21959

## B. 시뮬레이션 결과 요약 및 핵심 지표

### B-1) 전체 포트폴리오 요약 통계
 총 분석 고유 개인 수   평균 노력 점수   평균 할인율    평균 최종 보험료    평균 예상 손실   전체 총 예상 이익     평균 예상 이익
          100 163.162259 0.111831 54893.375107 6532.709815 4.836067e+06 48360.665292

### B-2) 클러스터(위험군)별 상세 분석
 Cluster  개인수    평균_노력점수   평균_할인율      평균_최종보험료      평균_예상이익       총_예상이익   이익기여도_%
       1   65 154.759901 0.103901  35288.359803 32576.530227 2.117474e+06 43.785057
       2   26 174.426777 0.122425  85633.755590 75207.691075 1.955400e+06 40.433686
       3    9 191.304022 0.138494 107679.608684 84799.121827 7.631921e+05 15.781257

### B-3) 주요 지표 분포 분석 (전체)
범위   지표  count         mean          std          min          p05          p25          p50          p75          p95          max
전체 노력점수    100   163.162259    34.349657   145.249400   145.515117   149.228850   149.715167   150.105817   244.484617   244.610600
전체  할인율    100     0.111831     0.032622     0.092886     0.093283     0.098843     0.099573     0.100159     0.188903     0.188942
전체 예상이익    100 48360.665292 21798.936884 30845.135270 32611.566520 32631.651001 32671.346574 75352.776342 81255.361598 89174.704995

### B-4) 주요 지표 분포 분석 (클러스터별)
       범위   지표  count         mean         std          min          p05          p25          p50          p75          p95          max
Cluster 1 노력점수     65   154.759901   23.177130   145.331067   145.520667   149.192333   149.562933   150.017867   225.473907   244.486200
Cluster 1 예상이익     65 32576.530227  457.094930 30845.135270 31198.690308 32622.411042 32648.779961 32670.669339 32887.159571 32898.306676
Cluster 1  할인율     65     0.103901    0.022064     0.093008     0.093291     0.098789     0.099344     0.100027     0.171174     0.188903
Cluster 2 노력점수     26   174.426777   43.386984   145.249400   145.492667   149.374367   149.840700   220.786817   244.501233   244.610600
Cluster 2 예상이익     26 75207.691075  613.164974 74292.589361 74294.141368 74557.699217 75364.593233 75432.847803 76000.011485 76035.447300
Cluster 2  할인율     26     0.122425    0.041219     0.092886     0.093249     0.099062     0.099761     0.166688     0.188908     0.188942
Cluster 3 노력점수      9   191.304022   50.440594   145.687667   147.104947   149.300933   149.877800   244.420600   244.562920   244.570733
Cluster 3 예상이익      9 84799.121827 4148.609166 81100.918799 81105.913372 81209.067868 81885.380231 89162.272399 89174.058058 89174.704995
Cluster 3  할인율      9     0.138494    0.047850     0.093541     0.095663     0.098951     0.099817     0.188883     0.188927     0.188930

### B-5) 정책 효과 및 가치 제안
 할인 적용 고객 비율   평균 할인액(원)  평균 리스크 절감액(원)  노력점수-할인율 상관계수  노력점수-예상이익 상관계수
         1.0 7323.524893    1371.821728       0.999634        0.351918


- **할인 적용 고객 비율**이 높을수록 행동 기반 인센티브가 실제로 작동하고 있음을 의미합니다.
- **평균 할인액**은 고객 체감 혜택의 크기를 보여주며, 상품 매력도 개선 근거가 됩니다.
- **노력점수-할인율/이익 상관**은 "노력의 가치"를 정량적으로 설명합니다.
- **평균 리스크 절감액**은 행동 변화가 예상 손실 감소로 이어지는 위험관리 효과를 보여줍니다.


## C. 클러스터링 단독 vs 클러스터링+할인 통계 분석

### C-1) 보험료 비교 통계 (bootstrap 단위)
 bootstrap_id  mean_premium_cluster_only  mean_premium_with_score  mean_diff_score_minus_cluster
            1                    62216.9             56325.922683                   -5890.977317
            2                    62216.9             56669.931171                   -5546.968829
            3                    62216.9             56536.793171                   -5680.106829
            4                    62216.9             56651.938002                   -5564.961998
            5                    62216.9             56596.867476                   -5620.032524
            6                    62216.9             56466.526371                   -5750.373629
            7                    62216.9             56435.472054                   -5781.427946
            8                    62216.9             56494.365406                   -5722.534594
            9                    62216.9             56404.071771                   -5812.828229
           10                    62216.9             56549.950010                   -5666.949990
           11                    62216.9             56439.034093                   -5777.865907
           12                    62216.9             56447.662341                   -5769.237659
           13                    62216.9             56460.510849                   -5756.389151
           14                    62216.9             56418.790321                   -5798.109679
           15                    62216.9             56426.067621                   -5790.832379
           16                    62216.9             56455.857272                   -5761.042728
           17                    62216.9             56372.317977                   -5844.582023
           18                    62216.9             56631.469788                   -5585.430212
           19                    62216.9             56451.047768                   -5765.852232
           20                    62216.9             56480.959070                   -5735.940930

#### 보험료 비교 검정 결과
 n_bootstrap_id  mean_diff(score-cluster)  bootstrap95ci_low  bootstrap95ci_high  paired_t_stat  paired_t_pvalue  wilcoxon_stat  wilcoxon_pvalue
           5000              -5757.236448       -5760.310825        -5754.190086   -3692.388156              0.0            0.0              0.0

### C-2) 보험사 손익 비교 통계 (bootstrap 단위)
 bootstrap_id  mean_profit_cluster_only  mean_profit_with_score  mean_diff_with_score_minus_cluster
            1              54312.368457            49669.600717                        -4642.767741
            2              54312.368457            50004.402822                        -4307.965635
            3              54312.368457            49855.130254                        -4457.238204
            4              54312.368457            49968.785873                        -4343.582584
            5              54312.368457            49939.750229                        -4372.618228
            6              54312.368457            49759.165450                        -4553.203008
            7              54312.368457            49783.484382                        -4528.884075
            8              54312.368457            49828.805144                        -4483.563313
            9              54312.368457            49702.258058                        -4610.110399
           10              54312.368457            49873.966789                        -4438.401668
           11              54312.368457            49752.105389                        -4560.263068
           12              54312.368457            49816.944809                        -4495.423649
           13              54312.368457            49806.157260                        -4506.211197
           14              54312.368457            49777.542403                        -4534.826055
           15              54312.368457            49742.900591                        -4569.467867
           16              54312.368457            49814.134542                        -4498.233915
           17              54312.368457            49728.808441                        -4583.560017
           18              54312.368457            49978.100548                        -4334.267910
           19              54312.368457            49757.010010                        -4555.358448
           20              54312.368457            49807.466123                        -4504.902334

#### 보험사 손익 비교 검정 결과
 n_bootstrap_id  mean_diff(with_score-cluster)  bootstrap95ci_low  bootstrap95ci_high  paired_t_stat  paired_t_pvalue  wilcoxon_stat  wilcoxon_pvalue
           5000                    -4524.56529       -4527.650727        -4521.540637   -2910.902682              0.0            0.0              0.0


- `mean_diff`가 음수이면 클러스터링+할인 모델이 평균 보험료(또는 평균 손익)를 낮춘 방향입니다.
- p-value가 작을수록 두 모델 차이가 우연이 아닐 가능성이 큽니다.
- 95% 신뢰구간에 0이 포함되지 않으면 모델 차이의 방향성이 더 안정적이라고 해석할 수 있습니다.

## D. 부트스트래핑 활용 목적 및 추가 분석

### D-1) 현재 코드의 부트스트래핑 활용 목적

1. 5000회 부트스트랩은 개인별 30일 행동 패턴의 변동성을 모사하기 위한 절차입니다.
2. 부트스트랩 결과를 평균해 `monthly_avg_total_score`를 안정적인 기대 행동 점수로 추정합니다.
3. 이를 통해 일시적 변동(노이즈)에 덜 민감한 보험료 산정 기준을 제공합니다.

### D-2) 고급 부트스트랩 불확실성 추정 (추가 구현)
                 지표           평균        표준편차    95% CI 하한    95% CI 상한
 부트스트랩별 포트폴리오 평균 이익 4.852494e+04   58.249041 4.841298e+04 4.864112e+04
부트스트랩별 포트폴리오 평균 할인율 1.118172e-01    0.001249 1.093480e-01 1.142695e-01
  부트스트랩별 포트폴리오 총 이익 4.852494e+06 5824.904050 4.841298e+06 4.864112e+06

#### 부트스트랩별 포트폴리오 통계 (일부)
 bootstrap_id  n_seqn  mean_profit  mean_discount_rate  mean_premium  mean_expected_loss  total_profit
            1     100 48489.228214            0.113034  54807.873292         6318.645078  4.848923e+06
            2     100 48602.777162            0.110577  55045.640215         6442.863053  4.860278e+06
            3     100 48533.116478            0.111195  54952.881221         6419.764743  4.853312e+06
            4     100 48642.169281            0.109205  55039.936258         6397.766977  4.864217e+06
            5     100 48626.916151            0.109942  54996.599045         6369.682894  4.862692e+06
            6     100 48518.599914            0.111351  54914.580834         6395.980920  4.851860e+06
            7     100 48551.213920            0.112645  54887.198958         6335.985039  4.855121e+06
            8     100 48511.237113            0.112029  54911.305713         6400.068600  4.851124e+06
            9     100 48493.646944            0.112319  54865.914480         6372.267537  4.849365e+06
           10     100 48525.770311            0.111735  54945.543732         6419.773421  4.852577e+06
           11     100 48463.912929            0.112493  54874.818999         6410.906070  4.846391e+06
           12     100 48565.410445            0.111429  54884.015942         6318.605497  4.856541e+06
           13     100 48546.451034            0.111591  54891.025811         6344.574777  4.854645e+06
           14     100 48534.806605            0.112389  54859.114063         6324.307458  4.853481e+06
           15     100 48495.581916            0.112821  54876.210589         6380.628673  4.849558e+06
           16     100 48559.148551            0.111889  54902.462444         6343.313893  4.855915e+06
           17     100 48505.488878            0.112473  54826.594863         6321.105985  4.850549e+06
           18     100 48627.917679            0.108679  55013.437470         6385.519791  4.862792e+06
           19     100 48469.328678            0.111988  54880.329301         6411.000622  4.846933e+06
           20     100 48512.948760            0.111705  54906.453476         6393.504716  4.851295e+06

#### 개인별 profit 부트스트랩 변동성 상위 20명
   SEQN    boot_mean    boot_std         q025         q975  n_boot
66444.0 82444.093067 1106.532033 81140.502637 85171.148115    5000
70186.0 76246.242914 1051.036814 74962.721492 78623.629106    5000
73884.0 76218.168633 1034.880146 74962.721492 78536.800963    5000
76657.0 76208.412388 1032.606835 74962.721492 78580.287087    5000
75738.0 76202.829135 1029.468954 74962.721492 78580.287087    5000
76954.0 76193.561991 1028.703508 74952.201176 78536.800963    5000
63602.0 82189.322638  889.495145 81120.049328 84489.048759    5000
83067.0 82167.443826  884.182792 81120.049328 84431.129662    5000
65653.0 75833.622045  880.261078 74856.729579 78004.284321    5000
67424.0 75828.505487  879.664176 74856.729579 78004.284321    5000
80936.0 75817.629712  868.428214 74835.423833 77914.861165    5000
77076.0 75821.097027  861.068998 74856.729579 77960.197178    5000
66708.0 75794.094711  853.945841 74856.729579 77959.066739    5000
65962.0 75781.395355  850.268530 74867.379374 77959.066739    5000
75527.0 82151.538764  849.891070 81120.049328 84373.054976    5000
70493.0 82147.367417  845.726687 81120.049328 84314.826990    5000
74782.0 75778.128510  843.014407 74867.379374 77913.727689    5000
68455.0 75760.877668  838.974967 74846.076986 77868.268958    5000
66749.0 75771.742305  836.278693 74846.076986 77869.405426    5000
65951.0 75790.525358  834.422715 74856.729579 77868.268958    5000



### 부록

In [13]:
# # 로지스틱 기반 할인율 계산 (MLE/최적화 미사용)
# # 할인율 = 5% * [시그모이드(E) / 시그모이드(만점)]
# import numpy as np
# import pandas as pd

# # 1) 전체 데이터 사용: bootstrap_row_id 기준 1행=1사람
# if 'Cluster' not in scored_df.columns:
#     if 'cluster_id' in scored_df.columns:
#         scored_df['Cluster'] = scored_df['cluster_id']
#     else:
#         raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

# required_person_cols = ['bootstrap_row_id', 'Cluster', 'total_score']
# missing_person_cols = [c for c in required_person_cols if c not in scored_df.columns]
# if missing_person_cols:
#     raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_person_cols}")

# base_cols = ['bootstrap_row_id', 'Cluster', 'total_score']
# if 'SEQN' in scored_df.columns:
#     base_cols.insert(1, 'SEQN')

# monthly = scored_df[base_cols].copy()
# monthly = monthly.rename(columns={'total_score': 'monthly_avg_total_score'})
# if monthly.empty:
#     raise ValueError("집계 가능한 데이터가 없습니다. scored_df를 확인하세요.")

# # 2) 기본보험료 및 λ0 매핑
# base_map = {1: P1, 2: p2, 3: p3}
# monthly['base_premium_P0'] = monthly['Cluster'].map(base_map)

# alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
# beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
# prevalence_by_cluster = {
#     1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
#     2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
#     3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
# }

# lambda_map = {
#     cluster: float(np.sum([
#         (prev_by_disease[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
#         for idx in alpha_by_disease
#     ]))
#     for cluster, prev_by_disease in prevalence_by_cluster.items()
# }
# monthly['lambda0'] = monthly['Cluster'].map(lambda_map)

# if monthly[['base_premium_P0', 'lambda0']].isna().any().any():
#     raise ValueError("기본보험료 또는 lambda0 매핑에 누락된 Cluster가 있습니다.")

# theta = float(np.mean([theta_step, theta_sleep, theta_sleep_pattern]))
# E_vals = monthly['monthly_avg_total_score'].clip(0, 300).to_numpy(dtype=float)
# base_vals = monthly['base_premium_P0'].to_numpy(dtype=float)
# lambda0_vals = monthly['lambda0'].to_numpy(dtype=float)

# FULL_SCORE = 300.0
# DISCOUNT_CAP = 0.05  # 만점자 할인율 5%

# # 3) 할인 함수 (고정 파라미터 사용, 최적화 없음)
# def sigmoid_score(E, k, E0):
#     E_arr = np.asarray(E, dtype=float)
#     return 1.0 / (1.0 + np.exp(-k * (E_arr - E0)))

# # 파라미터 셀의 로지스틱 파라미터를 그대로 사용 (k, E0)
# k_used = float(discount_k)
# E0_used = float(discount_E0)

# sigmoid_vals = sigmoid_score(E_vals, k_used, E0_used)
# sigmoid_full = float(sigmoid_score(np.array([FULL_SCORE]), k_used, E0_used)[0])

# if sigmoid_full <= 0:
#     raise ValueError("만점 시그모이드 값이 0 이하입니다. discount_k/discount_E0를 확인하세요.")

# # 요청식: cap * (해당 점수 로지스틱/시그모이드 값) / 만점 로지스틱/시그모이드 값
# d_opt = DISCOUNT_CAP * (sigmoid_vals / sigmoid_full)
# d_opt = np.clip(d_opt, 0.0, DISCOUNT_CAP)

# raw_discount_opt = base_vals * d_opt
# if max_discount_amount > 0:
#     applied_discount_opt = np.minimum(raw_discount_opt, max_discount_amount)
# else:
#     applied_discount_opt = raw_discount_opt

# premium_opt = base_vals - applied_discount_opt
# delta_E_opt = np.maximum(0.0, E_vals - E0_used) / 300.0
# lambda_eff_opt = lambda0_vals * np.exp(-theta * delta_E_opt)
# expected_loss_opt = lambda_eff_opt * severity + operating_cost
# profit_opt = premium_opt - expected_loss_opt

# full_score_mask = np.isclose(E_vals, FULL_SCORE)
# full_score_discount_mean = float(np.mean(d_opt[full_score_mask])) if np.any(full_score_mask) else np.nan

# print("=== 로지스틱 비율 기반 할인 결과 (MLE 미사용) ===")
# print("집계 기준: bootstrap_row_id 기준 1행=1사람(모든 행 독립)")
# print(f"사용 데이터 행 수 = {len(monthly):,}")
# print(f"사용 파라미터: k={k_used:.6f}, E0={E0_used:.3f}")
# print(f"할인율 식: d(E) = {DISCOUNT_CAP:.2%} * sigmoid(E) / sigmoid(300)")
# print(f"만점자 수(E=300) = {int(full_score_mask.sum())}")
# if np.any(full_score_mask):
#     print(f"만점자 평균 할인율 = {full_score_discount_mean:.2%}")
# else:
#     print("만점자 데이터가 없어 만점 할인율 실측값은 계산되지 않았습니다.")
# print(f"평균 이익(보험사) = {float(np.mean(profit_opt)):.2f}")
# print(f"평균 보험료(고객) = {float(np.mean(premium_opt)):.2f}")
# print(f"평균 할인비율    = {float(np.mean(applied_discount_opt / base_vals)):.4f}")

# monthly_sample = monthly.copy()
# monthly_sample['discount_rate'] = d_opt
# monthly_sample['applied_discount_amount'] = applied_discount_opt
# monthly_sample['premium_with_score'] = premium_opt
# monthly_sample['delta_E'] = delta_E_opt
# monthly_sample['lambda_effective'] = lambda_eff_opt
# monthly_sample['expected_loss'] = expected_loss_opt
# monthly_sample['profit'] = profit_opt

# id_cols = ['bootstrap_row_id']
# if 'SEQN' in monthly_sample.columns:
#     id_cols.append('SEQN')
# id_cols.append('Cluster')

# monthly_sample[
#     id_cols + [
#         'monthly_avg_total_score', 'base_premium_P0',
#         'discount_rate', 'applied_discount_amount', 'premium_with_score',
#         'lambda0', 'lambda_effective', 'expected_loss', 'profit'
#     ]
# ].head(500)